<a href="https://colab.research.google.com/github/samer-glitch/TADP-Cluster-Computing/blob/main/01_TADP_Experiment_A_A1_Main_Benchmark_K10_5Seed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
import io
import os

print("📤 Please upload the diabetes_130US.csv file:")
uploaded = files.upload()
DATA_FILENAME  = "diabetes_130US.csv"
DATA_CACHE_DIR = "./data_cache"
os.makedirs(DATA_CACHE_DIR, exist_ok=True)

# Get the uploaded file
for filename in uploaded.keys():
    file_data = uploaded[filename]
    print(f'✅ Uploaded: {filename} ({len(file_data)} bytes)')


    # Save to cache
    cache_path = os.path.join(DATA_CACHE_DIR, DATA_FILENAME)
    with open(cache_path, 'wb') as f:
        f.write(file_data)
    print(f'✅ Saved to cache: {cache_path}')

    # Verify the file was saved
    if os.path.exists(cache_path):
        file_size = os.path.getsize(cache_path)
        print(f'✅ Verified: {cache_path} exists ({file_size} bytes)')
    else:
        print(f'❌ Error: File was not saved properly')

📤 Please upload the diabetes_130US.csv file:


Saving diabetes_130US.csv to diabetes_130US.csv
✅ Uploaded: diabetes_130US.csv (19159383 bytes)
✅ Saved to cache: ./data_cache/diabetes_130US.csv
✅ Verified: ./data_cache/diabetes_130US.csv exists (19159383 bytes)


In [2]:
# ======================================================================================
# 01_TADP_Experiment_A_A1_Main_Benchmark_K10_5Seed_COLAB.py
# Colab-ready Python export
# ======================================================================================


# %% [CELL 1 - MARKDOWN]
# # TADP Experiment A — A1 Main Benchmark (K=10, 5 Seeds)
#
# **Final governance design:** 28-factor evidence-validity gate → six-dimension floor → six-dimension HPS → four-critical-dimension normalized `HPS_review` for the borderline band.
#
# **Option A pipeline:** raw TRAIN shards → DQ/governance → Accept/Reject → fit scaler, categorical vocabulary, and class weights from the participating TRAIN cohort only → training → held-out TEST evaluation.
#
# <a href="https://colab.research.google.com/github/samer-glitch/TADP-Cluster-Computing/blob/main/01_TADP_Experiment_A_A1_Main_Benchmark_K10_5Seed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# %% [CELL 2 - CODE]
import os

# Dataset upload is intentionally handled in a SEPARATE Colab cell before this script.
# This experiment never opens a second upload dialog.
DATA_FILENAME = "diabetes_130US.csv"
DATA_CACHE_DIR = "./data_cache"
DATA_CACHE_PATH = os.path.join(DATA_CACHE_DIR, DATA_FILENAME)

if os.path.exists(DATA_CACHE_PATH):
    print(
        f"✅ Reusing dataset already cached by the separate upload cell: "
        f"{os.path.abspath(DATA_CACHE_PATH)}"
    )
else:
    print(
        "ℹ️ Dataset is not cached yet. Run the separate upload cell first; "
        "it must save diabetes_130US.csv as ./data_cache/diabetes_130US.csv."
    )

# %% [CELL 3 - CODE]
# ======================================================================================
# TADP — EXPERIMENT A1 MAIN BENCHMARK | K=10 | 5 TRAINING SEEDS | 4 FL ROUNDS
# ======================================================================================
# FINAL GOVERNANCE DESIGN USED IN THIS NOTEBOOK
#
# Governance is evaluated BEFORE model preprocessing:
#   raw GLOBAL TRAIN -> raw client TRAIN shards -> DQ + documentary governance
#   -> final Accept/Reject -> fit model scaler/vocabulary/class weights from the
#   participating TRAIN cohort only -> train -> evaluate once on held-out TEST.
#
# The held-out TEST set is created before client partitioning and is never used for
# DQ scoring, governance, participant selection, preprocessing, class weights,
# optimizer-step budgeting, model initialization, or training.
#
# TADP admission policy:
#   1) Evidence-validity gate over ALL 28 factor scores:
#        every score must be present, finite, and strictly > 0.
#        Any missing, non-finite, or zero/non-positive score -> AUTO-REJECT.
#   2) Dimension floor:
#        every one of the six averaged HPS dimensions must be >= 2.5/5.
#   3) Main HPS over all six dimensions:
#        HPS < 3.0  -> AUTO-REJECT
#        HPS >= 3.5 -> DIRECT AUTO-ACCEPT
#        3.0 <= HPS < 3.5 -> AUTOMATED REVIEW
#   4) Automated Review:
#        critical dimensions = Source Reliability (dim1),
#                              Data Quality and Health (dim2),
#                              Regulatory/Compliance Alignment (dim5),
#                              Context/Usage Constraints (dim6).
#        Documentation Practices (dim3) and Timeliness (dim4) remain part of the
#        main HPS but are intentionally excluded from the borderline review score.
#        HPS_review reuses the original HPS weights, renormalized over the four
#        critical dimensions only.
#        HPS_review >= midpoint(3.0, 3.5)=3.25 -> ACCEPT AFTER REVIEW
#        otherwise                            -> AUTO-REJECT.
#
# No factor-specific healthcare mandatory minima and no separate 14-factor Review
# Score are used in this notebook.
# ======================================================================================

import os


import sys
import gc
import math
import time
import json
import random
import hashlib
import threading
import zipfile
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


# ======================================================================================
# POLICY CONSTANTS — FINAL A1 DESIGN
# ======================================================================================

GOOD_CUT = 3.0
HIGH_CUT = 3.5
MAX_FACTOR_SCORE = 5.0
GE_ACCEPT_COUNT = 6

DIMENSION_MIN_FLOOR = 2.5
REVIEW_MIDPOINT = (GOOD_CUT + HIGH_CUT) / 2.0  # 3.25
EVIDENCE_VALIDITY_MIN_EXCLUSIVE = 0.0

# Four critical dimensions used only to resolve the HPS borderline band.
# The original HPS weights are renormalized over this set.
CRITICAL_REVIEW_DIMS = ("dim1", "dim2", "dim5", "dim6")

WEIGHTS_PSCORE_DEFAULT = {


    "dim1": 0.25,  # Source Reliability
    "dim2": 0.15,  # Data Quality and Health
    "dim3": 0.10,  # Documentation Practices
    "dim4": 0.10,  # Timeliness and Refresh Rate
    "dim5": 0.30,  # Regulatory / Compliance Alignment
    "dim6": 0.10,  # Context / Usage Constraints
}



# Renormalize the ORIGINAL six-dimension HPS weights over the four
# policy-defined critical dimensions used only in the automated-review band.
_CRITICAL_REVIEW_WEIGHT_TOTAL = sum(
    WEIGHTS_PSCORE_DEFAULT[d] for d in CRITICAL_REVIEW_DIMS
)
CRITICAL_REVIEW_WEIGHTS = {
    d: WEIGHTS_PSCORE_DEFAULT[d] / _CRITICAL_REVIEW_WEIGHT_TOTAL
    for d in CRITICAL_REVIEW_DIMS
}

# Fail closed if the review weights are not a valid normalized distribution.
assert _CRITICAL_REVIEW_WEIGHT_TOTAL > 0.0
assert abs(sum(CRITICAL_REVIEW_WEIGHTS.values()) - 1.0) < 1e-12

DIMENSION_NAMES = {
    "dim1": "Source Reliability",
    "dim2": "Data Quality and Health",
    "dim3": "Documentation Practices",
    "dim4": "Timeliness and Refresh Rate",
    "dim5": "Regulatory and Compliance Alignment",
    "dim6": "Context and Usage Constraints",
}

# Factor-count notes:
#   dim1: data_collection_lineage
#   dim2: structural_constraint_integrity
#
# Total = 4 + 8 + 4 + 3 + 5 + 4 = 28 factors.
FACTOR_NAMES = {
    "dim1": [
        "source_reputation",
        "data_controller",
        "data_objective",
        "data_collection_lineage",
    ],
    "dim2": [
        "completeness",
        "duplication_rate",
        "value_validity_error_rate",
        "type_consistency",
        "label_integrity",
        "feature_distribution_consistency",
        "feature_category_coverage",
        "structural_constraint_integrity",
    ],
    "dim3": [
        "data_dictionary",
        "version_logs",
        "collection_protocol",
        "definition_updates",
    ],
    "dim4": [
        "data_freshness",
        "scheduled_refresh",
        "retention_clarity",
    ],
    "dim5": [
        "regulation_coverage",
        "consent_ethics",
        "geo_restrictions",
        "sensitivity_classification",
        "audits",
    ],
    "dim6": [
        "license_terms",
        "ethical_reviews",
        "redistribution",
        "user_agreements",
    ],
}

DOCUMENTARY_DIMS = ("dim1", "dim3", "dim4", "dim5", "dim6")

# ----------------------------------------------------------------------
# COMPLETE 0--5 RUBRIC DESCRIPTORS
# ----------------------------------------------------------------------
# These descriptors reproduce the Appendix-A semantics and add the two
# two reviewer-driven factors explicitly. Controlled evidence is sampled at the factor
# level; HPS is never generated directly.
RUBRIC_DESCRIPTORS = {
    "dim1": {
        "source_reputation": {
            0: "No info",
            1: "Poor",
            2: "Limited evidence",
            3: "Average, partially trusted",
            4: "Well-documented, reliable",
            5: "Highly reputable, verified",
        },
        "data_controller": {
            0: "No documented controller",
            1: "Unclear",
            2: "Partially clear",
            3: "Moderately clear",
            4: "Mostly clear",
            5: "Fully documented",
        },
        "data_objective": {
            0: "None",
            1: "Vague",
            2: "Partial",
            3: "General but unclear",
            4: "Mostly explicit",
            5: "Fully explicit, justified",
        },
        "data_collection_lineage": {
            0: "Collection origin unknown",
            1: "Informal or unverifiable origin",
            2: "Partially documented acquisition path",
            3: "Documented acquisition with limited traceability",
            4: "Well-documented and traceable acquisition path",
            5: "Fully source-linked, versioned, and auditable lineage",
        },
    },
    "dim2": {
        "completeness": {
            0: ">50% missing",
            1: "20-50% missing",
            2: "10-20% missing",
            3: "5-10% missing",
            4: "1-5% missing",
            5: "<1% missing",
        },
        "duplication_rate": {
            0: ">20% duplicates",
            1: "10-20% duplicates",
            2: "5-10% duplicates",
            3: "2-5% duplicates",
            4: "1-2% duplicates",
            5: "<1% duplicates",
        },
        "value_validity_error_rate": {
            0: ">15% invalid/error values",
            1: "10-15% invalid/error values",
            2: "5-10% invalid/error values",
            3: "2-5% invalid/error values",
            4: "1-2% invalid/error values",
            5: "<1% invalid/error values",
        },
        "type_consistency": {
            0: "Highly inconsistent",
            1: "Frequent type inconsistency",
            2: "Moderate type inconsistency",
            3: "Minor type inconsistency",
            4: "Rare type inconsistency",
            5: "Fully consistent",
        },
        "label_integrity": {
            0: ">10% missing/invalid/known erroneous labels",
            1: "5-10% missing/invalid/known erroneous labels",
            2: "2-5% missing/invalid/known erroneous labels",
            3: "1-2% missing/invalid/known erroneous labels",
            4: "0.1-1% missing/invalid/known erroneous labels",
            5: "<=0.1% missing/invalid/known erroneous labels",
        },
        "feature_distribution_consistency": {
            0: "JSD >0.20",
            1: "JSD 0.10-0.20",
            2: "JSD 0.05-0.10",
            3: "JSD 0.025-0.05",
            4: "JSD 0.01-0.025",
            5: "JSD <=0.01",
        },
        "feature_category_coverage": {
            0: "<50% reference support represented",
            1: "50-65% reference support represented",
            2: "65-75% reference support represented",
            3: "75-82.5% reference support represented",
            4: "82.5-90% reference support represented",
            5: ">=90% reference support represented",
        },
        "structural_constraint_integrity": {
            0: ">10% records/structures violate required constraints",
            1: "5-10% violate required constraints",
            2: "2-5% violate required constraints",
            3: "1-2% violate required constraints",
            4: "0.1-1% violate required constraints",
            5: "<=0.1% violate required constraints",
        },
    },
    "dim3": {
        "data_dictionary": {
            0: "None",
            1: "Minimal outline",
            2: "Partial coverage",
            3: "Moderate coverage",
            4: "Near-complete",
            5: "Fully detailed",
        },
        "version_logs": {
            0: "None",
            1: "Minimal logs",
            2: "Occasional logs",
            3: "Regular logs",
            4: "Near-complete",
            5: "Full version history",
        },
        "collection_protocol": {
            0: "None",
            1: "Vague",
            2: "Partial",
            3: "General methods",
            4: "Well-defined",
            5: "Fully transparent",
        },
        "definition_updates": {
            0: "None",
            1: "Rarely updated",
            2: "Occasional updates",
            3: "Regular but basic",
            4: "Frequent",
            5: "Real-time, documented",
        },
    },
    "dim4": {
        "data_freshness": {
            0: ">5 years old",
            1: "2-5 years old",
            2: "1-2 years old",
            3: "6-12 months old",
            4: "1-6 months old",
            5: "Real-time/current",
        },
        "scheduled_refresh": {
            0: "Never",
            1: "Irregular",
            2: "Annual",
            3: "Quarterly",
            4: "Monthly",
            5: "Daily/real-time",
        },
        "retention_clarity": {
            0: "None",
            1: "Minimal",
            2: "Basic guidelines",
            3: "Moderate clarity",
            4: "High clarity",
            5: "Fully documented",
        },
    },
    "dim5": {
        "regulation_coverage": {
            0: "None",
            1: "Minimal",
            2: "Partial",
            3: "Moderate",
            4: "Comprehensive but dated",
            5: "Fully documented/current",
        },
        "consent_ethics": {
            0: "None",
            1: "Minimal record",
            2: "Partial consent/ethics evidence",
            3: "Moderate logs",
            4: "Substantial",
            5: "Fully documented",
        },
        "geo_restrictions": {
            0: "None",
            1: "Basic mention",
            2: "Partial",
            3: "Moderate",
            4: "Near-complete",
            5: "Fully documented",
        },
        "sensitivity_classification": {
            0: "None",
            1: "Basic flagging",
            2: "Partial",
            3: "Moderate",
            4: "Near-complete",
            5: "Fully classified",
        },
        "audits": {
            0: "None",
            1: "Internal only",
            2: "Basic certification",
            3: "Occasional audit",
            4: "Recent audit",
            5: "Regular external audits",
        },
    },
    "dim6": {
        "license_terms": {
            0: "None",
            1: "Vague",
            2: "Basic",
            3: "Clear",
            4: "Detailed",
            5: "Industry-compliant",
        },
        "ethical_reviews": {
            0: "None",
            1: "Informal approval",
            2: "Partial",
            3: "Moderate",
            4: "Well-documented",
            5: "Certified",
        },
        "redistribution": {
            0: "No policy",
            1: "Unclear",
            2: "Partial",
            3: "Clear",
            4: "Detailed",
            5: "Fully compliant",
        },
        "user_agreements": {
            0: "Non-compliant",
            1: "Minimal adherence",
            2: "Partial",
            3: "Mostly compliant",
            4: "Fully compliant",
            5: "Audited compliance",
        },
    },
}

# ----------------------------------------------------------------------
# FACTOR POLICY NOTE
# ----------------------------------------------------------------------
# The 0--5 rubric definitions above provide the scoring semantics.
# Admission does NOT use separate factor-specific adequacy minima.
# All 28 scores are first checked only for evidence validity
# (present, finite, and >0), followed by the six-dimension floor and HPS rules.

DQ_RAW_METRIC_BY_FACTOR = {


    "completeness": "missing_fraction",
    "duplication_rate": "duplicate_fraction",
    "value_validity_error_rate": "error_fraction",
    "type_consistency": "type_inconsistency_fraction",
    "label_integrity": "invalid_label_fraction",
    "feature_distribution_consistency": "max_jsd",
    "feature_category_coverage": "mean_category_coverage",
    "structural_constraint_integrity": "structural_violation_fraction",
}

assert abs(sum(WEIGHTS_PSCORE_DEFAULT.values()) - 1.0) < 1e-12
assert sum(len(v) for v in FACTOR_NAMES.values()) == 28
assert len(FACTOR_NAMES["dim1"]) == 4
assert len(FACTOR_NAMES["dim2"]) == 8

# ======================================================================================
# GENERAL UTILITIES
# ======================================================================================

def seed_everything(seed: int):
    seed = int(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        tf.keras.utils.set_random_seed(seed)
    except Exception:
        tf.random.set_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def sha256_bytes(x: bytes) -> str:
    return hashlib.sha256(x).hexdigest()


def sha256_array(x: np.ndarray) -> str:
    x = np.asarray(x)
    return sha256_bytes(np.ascontiguousarray(x).view(np.uint8).tobytes())


def sha256_weights(weights: List[np.ndarray]) -> str:
    h = hashlib.sha256()
    for w in weights:
        a = np.ascontiguousarray(np.asarray(w))
        h.update(str(a.shape).encode())
        h.update(a.view(np.uint8).tobytes())
    return h.hexdigest()


def _process_rss_mb() -> float:
    try:
        import psutil
        return float(psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2))
    except Exception:
        pass
    try:
        with open("/proc/self/statm", "r", encoding="utf-8") as f:
            pages = int(f.read().split()[1])
        return float(pages * int(os.sysconf("SC_PAGE_SIZE")) / (1024 ** 2))
    except Exception:
        pass
    try:
        import resource
        x = float(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
        return x / (1024 ** 2) if sys.platform == "darwin" else x / 1024.0
    except Exception:
        return 0.0


class RAMMonitor:
    def __init__(self, interval_s: float = 0.05):
        self.interval_s = float(interval_s)
        self.start_mb = 0.0
        self.end_mb = 0.0
        self.peak_mb = 0.0
        self._stop = threading.Event()
        self._thread = None

    def _loop(self):
        while not self._stop.wait(self.interval_s):
            self.peak_mb = max(self.peak_mb, _process_rss_mb())

    def start(self):
        self.start_mb = _process_rss_mb()
        self.peak_mb = self.start_mb
        self._stop.clear()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        return self

    def stop(self) -> Dict[str, float]:
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=1.0)
        self.end_mb = _process_rss_mb()
        self.peak_mb = max(self.peak_mb, self.start_mb, self.end_mb)
        return {
            "ram_start_mb": float(self.start_mb),
            "ram_end_mb": float(self.end_mb),
            "ram_peak_mb": float(self.peak_mb),
            "ram_delta_mb": float(max(0.0, self.peak_mb - self.start_mb)),
            "ram_mb": float(self.peak_mb),
        }


def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)
    return Path(path)


def append_csv(row: Dict[str, Any], path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([row]).to_csv(
        path, mode="a", header=not path.exists(), index=False
    )


def summarize_runs(perf: pd.DataFrame) -> pd.DataFrame:
    if perf.empty:
        return pd.DataFrame()
    metrics = [
        c for c in [
            "accuracy", "precision_macro", "recall_macro", "f1_macro",
            "roc_auc_ovr_macro", "runtime_s", "energy_wh_per_watt",
            "energy_wh", "communication_payload_bytes", "communication_mb",
            "optimizer_steps", "participants", "ram_peak_mb", "ram_delta_mb"
        ] if c in perf.columns
    ]
    rows = []
    for scenario, d in perf.groupby("scenario", sort=False):
        row = {"scenario": scenario, "n_runs": len(d)}
        for m in metrics:
            vals = pd.to_numeric(d[m], errors="coerce")
            row[f"{m}_mean"] = float(vals.mean())
            row[f"{m}_sd"] = float(vals.std(ddof=1)) if len(vals) > 1 else 0.0
        rows.append(row)
    return pd.DataFrame(rows)


def package_and_download_results(output_dir: Path, label: str) -> Path:
    output_dir = Path(output_dir)
    manifest = []
    for p in sorted(output_dir.rglob("*")):
        if p.is_file():
            manifest.append({
                "relative_path": str(p.relative_to(output_dir)),
                "bytes": int(p.stat().st_size),
                "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
            })
    pd.DataFrame(manifest).to_csv(
        output_dir / "RESULTS_FILE_MANIFEST.csv", index=False
    )

    zip_path = output_dir.parent / f"{output_dir.name}_RESULTS.zip"
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(
        zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6
    ) as zf:
        for p in sorted(output_dir.rglob("*")):
            if p.is_file():
                zf.write(p, arcname=str(p.relative_to(output_dir)))

    print("\n" + "=" * 100)
    print(f"{label} COMPLETE")
    print(f"Results ZIP: {zip_path}")
    print(f"ZIP size: {zip_path.stat().st_size / (1024**2):.2f} MB")
    print("=" * 100)

    try:
        from google.colab import files as colab_files
        print("Starting automatic download to your laptop...")
        colab_files.download(str(zip_path))
    except Exception as exc:
        print("Automatic Colab download unavailable.")
        print(f"ZIP remains at: {zip_path}")
        print(f"Reason: {exc}")

    return zip_path


# ======================================================================================
# CONTROLLED DOCUMENTARY EVIDENCE — FINAL A1 DESIGN
# ======================================================================================

def rubric_descriptor(dim: str, factor: str, score: float) -> str:
    s = int(np.clip(np.rint(float(score)), 0, 5))
    return str(RUBRIC_DESCRIPTORS[dim][factor][s])


def generate_controlled_documentary_evidence(
    client_ids: List[str],
    evidence_seed: int,
    domain: str,
) -> Tuple[Dict[str, Dict[str, Dict[str, float]]], pd.DataFrame]:
    """
    Generate controlled documentary evidence at the INDIVIDUAL FACTOR level.

    The documentary profiles are designed only to exercise branches of the FINAL
    A1 governance policy. Data Quality (dim2) is never synthesized here; its eight
    factors are measured from each raw TRAIN shard before admission.

    Archetypes:
      DIRECT_STRONG
          Strong evidence intended for direct HPS acceptance.
      REVIEW_RECOVERABLE
          Borderline main HPS but strong enough critical-dimension HPS_review.
      REVIEW_LIMITED
          Borderline main HPS but weak critical-dimension HPS_review.
      EVIDENCE_INVALID
          Otherwise strong documentary evidence with one deliberate zero score,
          exercising the all-28-factor validity gate.
      LOW_HPS_WEAK
          Every documentary dimension remains >=2.5 but overall HPS is designed
          to remain below 3.0 even with strong TRAIN-measured DQ.
      DIMENSION_FLOOR_WEAK
          Timeliness is deliberately below 2.5 while all documentary scores remain
          finite and strictly positive.

    These are controlled governance scenarios, not observed hospital-site
    provenance records or estimates of real-world prevalence. Assignment is frozen
    before model training and never uses TEST or downstream predictive performance.
    """
    domain = str(domain).lower()
    if domain not in {"healthcare", "cifar10"}:
        raise ValueError(f"Unsupported domain: {domain!r}")

    n = len(client_ids)
    rng = np.random.default_rng(int(evidence_seed))

    def set_values(factors, dim, values):
        if len(values) != len(FACTOR_NAMES[dim]):
            raise RuntimeError(f"Profile length mismatch for {dim}.")
        for factor, value in zip(FACTOR_NAMES[dim], values):
            factors[dim][factor] = float(value)

    def make_documentary_profile(role: str, variant: int):
        factors = {dim: {} for dim in DOCUMENTARY_DIMS}

        if role == "DIRECT_STRONG":
            for dim in DOCUMENTARY_DIMS:
                for j, factor in enumerate(FACTOR_NAMES[dim]):
                    factors[dim][factor] = float(
                        5.0 if ((j + variant + len(dim)) % 3 == 0) else 4.0
                    )

        elif role == "REVIEW_RECOVERABLE":
            # All documentary dimensions are centered at 3.0. With the measured
            # DQ profile in this dataset, this is intended to enter the main HPS
            # review band while the critical HPS_review can exceed 3.25.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    factors[dim][factor] = 3.0
            if variant % 2 == 1:
                factors["dim3"]["data_dictionary"] = 4.0
                factors["dim4"]["scheduled_refresh"] = 4.0

        elif role == "REVIEW_LIMITED":
            # Critical documentary dimensions are kept near the 2.5 floor while
            # Documentation and Timeliness are strong. This creates the intended
            # case where the global six-dimension HPS can reach the review band
            # but the renormalized critical HPS_review remains below the midpoint.
            set_values(factors, "dim1", [2.0, 3.0, 2.0, 3.0])          # mean 2.5
            set_values(factors, "dim3", [5.0, 5.0, 5.0, 5.0])          # mean 5.0
            set_values(factors, "dim4", [5.0, 5.0, 5.0])               # mean 5.0
            set_values(factors, "dim5", [2.0, 3.0, 2.0, 3.0, 3.0])     # mean 2.6
            set_values(factors, "dim6", [2.0, 3.0, 2.0, 3.0])          # mean 2.5

        elif role == "EVIDENCE_INVALID":
            # Strong evidence except for one explicit zero. The validity gate must
            # reject before dimensions or HPS are used for admission.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    factors[dim][factor] = 4.0
            invalid_candidates = [
                ("dim1", "source_reputation"),
                ("dim3", "data_dictionary"),
                ("dim4", "data_freshness"),
                ("dim5", "geo_restrictions"),
                ("dim6", "ethical_reviews"),
            ]
            weak_dim, weak_factor = invalid_candidates[
                int(variant) % len(invalid_candidates)
            ]
            factors[weak_dim][weak_factor] = 0.0

        elif role == "LOW_HPS_WEAK":
            # Every documentary dimension is exactly or slightly above 2.5.
            # Even if DQ is very strong, the six-dimension HPS stays below 3.0.
            set_values(factors, "dim1", [2.0, 3.0, 2.0, 3.0])          # 2.5
            set_values(factors, "dim3", [2.0, 3.0, 2.0, 3.0])          # 2.5
            set_values(factors, "dim4", [2.0, 3.0, 3.0])               # 2.667
            set_values(factors, "dim5", [2.0, 3.0, 2.0, 3.0, 3.0])     # 2.6
            set_values(factors, "dim6", [2.0, 3.0, 2.0, 3.0])          # 2.5

        elif role == "DIMENSION_FLOOR_WEAK":
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    factors[dim][factor] = 3.0
            for factor in FACTOR_NAMES["dim4"]:
                factors["dim4"][factor] = 2.0

        else:
            raise ValueError(f"Unknown evidence role: {role!r}")

        return factors

    # K=10 branch-coverage design. The mix is not used to force an acceptance K;
    # the actual admitted cohort is determined only by the policy after DQ is measured.
    if n == 10:
        bundle_specs = [
            ("DIRECT_STRONG", 0),
            ("DIRECT_STRONG", 1),
            ("DIRECT_STRONG", 2),
            ("DIRECT_STRONG", 3),
            ("REVIEW_RECOVERABLE", 0),
            ("REVIEW_RECOVERABLE", 1),
            ("EVIDENCE_INVALID", 0),
            ("REVIEW_LIMITED", 0),
            ("LOW_HPS_WEAK", 0),
            ("DIMENSION_FLOOR_WEAK", 0),
        ]
    else:
        archetypes = [
            "DIRECT_STRONG",
            "REVIEW_RECOVERABLE",
            "EVIDENCE_INVALID",
            "REVIEW_LIMITED",
            "LOW_HPS_WEAK",
            "DIMENSION_FLOOR_WEAK",
        ]
        bundle_specs = [
            (archetypes[j % len(archetypes)], j // len(archetypes))
            for j in range(n)
        ]

    bundles = []
    for b, (role, variant) in enumerate(bundle_specs):
        bundles.append({
            "bundle_id": f"E{b+1:02d}",
            "profile": role,
            "scenario_role": role,
            "profile_variant": int(variant),
            "factors": make_documentary_profile(role, int(variant)),
        })

    assignment = rng.permutation(n)
    evidence_by_client = {}
    rows = []

    for client_pos, cid in enumerate(client_ids):
        bundle = bundles[int(assignment[client_pos])]
        evidence_by_client[cid] = {
            dim: dict(bundle["factors"][dim])
            for dim in DOCUMENTARY_DIMS
        }

        for dim in DOCUMENTARY_DIMS:
            for factor, score in bundle["factors"][dim].items():
                rows.append({
                    "client": str(cid),
                    "bundle_id": bundle["bundle_id"],
                    "evidence_profile": bundle["profile"],
                    "scenario_role": bundle["scenario_role"],
                    "profile_variant": int(bundle["profile_variant"]),
                    "dimension": dim,
                    "dimension_name": DIMENSION_NAMES[dim],
                    "factor": factor,
                    "evidence_source_type": "CONTROLLED_DOCUMENTARY_EVIDENCE",
                    "verified_rubric_level_0_5": float(score),
                    "rubric_score_0_5": float(score),
                    "server_mapped_score_0_5": float(score),
                    "rubric_descriptor": rubric_descriptor(dim, factor, score),
                    "human_role": "VERIFY_EVIDENCE_ONLY",
                    "score_assignment": "DETERMINISTIC_SERVER_MAPPING_FROM_VERIFIED_RUBRIC_LEVEL",
                    "admission_decision_by": "SERVER_POLICY",
                    "evidence_validity_rule": "present_finite_and_strictly_gt_0",
                    "factor_valid_for_gate": bool(np.isfinite(score) and float(score) > 0.0),
                    "is_critical_review_dimension": bool(dim in CRITICAL_REVIEW_DIMS),
                    "evidence_artifact_id": f"{cid}-{bundle['bundle_id']}-{dim}-{factor}",
                    "validation_status": "CONTROLLED_SCENARIO_EVIDENCE",
                    "evidence_seed": int(evidence_seed),
                    "domain": domain,
                })

    return evidence_by_client, pd.DataFrame(rows)

def dimension_scores_from_factors(
    factors: Dict[str, Dict[str, float]]
) -> Dict[str, float]:
    """Mean 0--5 factor score within each of the six HPS dimensions."""
    out = {}
    for dim, expected_factors in FACTOR_NAMES.items():
        vals = []
        valid = True
        for factor in expected_factors:
            if factor not in factors.get(dim, {}):
                valid = False
                break
            try:
                value = float(factors[dim][factor])
            except Exception:
                valid = False
                break
            if not np.isfinite(value):
                valid = False
                break
            vals.append(value)
        out[dim] = float(np.mean(vals)) if valid and vals else np.nan
    return out


def compute_hps(
    dimensions: Dict[str, float],
    weights: Dict[str, float] = WEIGHTS_PSCORE_DEFAULT,
) -> float:
    vals = [float(dimensions[d]) for d in weights]
    if not all(np.isfinite(v) for v in vals):
        return np.nan
    return float(sum(float(weights[d]) * float(dimensions[d]) for d in weights))


def evidence_validity_audit(
    factors: Dict[str, Dict[str, float]]
) -> Dict[str, Any]:
    """
    Gate 1: every one of the 28 expected factor scores must be present, finite,
    and strictly greater than zero. No factor-specific adequacy minima are used.
    """
    missing, nonfinite, nonpositive = [], [], []
    scores = {}

    for dim, expected_factors in FACTOR_NAMES.items():
        for factor in expected_factors:
            key = f"{dim}.{factor}"
            if factor not in factors.get(dim, {}):
                missing.append(key)
                continue
            try:
                value = float(factors[dim][factor])
            except Exception:
                nonfinite.append(key)
                continue
            if not np.isfinite(value):
                nonfinite.append(key)
                continue
            scores[key] = value
            if value <= EVIDENCE_VALIDITY_MIN_EXCLUSIVE:
                nonpositive.append(f"{key}:{value:.3f}")

    expected_n = int(sum(len(v) for v in FACTOR_NAMES.values()))
    passed = (
        len(scores) == expected_n
        and not missing
        and not nonfinite
        and not nonpositive
    )
    return {
        "passed": bool(passed),
        "factor_count_expected": expected_n,
        "factor_count_observed_valid_numeric": int(len(scores)),
        "missing": missing,
        "nonfinite": nonfinite,
        "nonpositive": nonpositive,
    }


def dimension_floor_failures(
    dimensions: Dict[str, float],
    floor: float = DIMENSION_MIN_FLOOR,
) -> List[str]:
    return [
        dim
        for dim, value in dimensions.items()
        if (not np.isfinite(float(value))) or float(value) < float(floor)
    ]


def compute_hps_review(
    dimensions: Dict[str, float]
) -> float:
    """
    Borderline review score using the SAME HPS dimension weights renormalized
    over the four policy-defined critical dimensions only:
      dim1 Source Reliability
      dim2 Data Quality and Health
      dim5 Regulatory/Compliance Alignment
      dim6 Context/Usage Constraints
    """
    vals = [float(dimensions[d]) for d in CRITICAL_REVIEW_DIMS]
    if not all(np.isfinite(v) for v in vals):
        return np.nan
    return float(
        sum(
            CRITICAL_REVIEW_WEIGHTS[d] * float(dimensions[d])
            for d in CRITICAL_REVIEW_DIMS
        )
    )


def tadp_decision(
    factors: Dict[str, Dict[str, float]],
    domain: str,
    good_cut: float = GOOD_CUT,
    high_cut: float = HIGH_CUT,
) -> Dict[str, Any]:
    """
    FINAL A1 TADP admission policy.

    1) All 28 factor scores present, finite, and >0; otherwise Reject.
    2) Every dimension >=2.5; otherwise Reject.
    3) Main HPS over all six dimensions:
         HPS <3.0  -> Reject
         HPS >=3.5 -> Direct Accept
         [3.0,3.5) -> Automated Review
    4) Review uses HPS_review over dim1, dim2, dim5, dim6 with original HPS
       weights renormalized over those four dimensions.
         HPS_review >3.25 -> Accept after Review
         otherwise        -> Reject
    """
    domain = str(domain).lower()

    validity = evidence_validity_audit(factors)
    dims = dimension_scores_from_factors(factors)
    hps = compute_hps(dims)
    hps_review = compute_hps_review(dims)

    dim_floor_failed = dimension_floor_failures(
        dims, floor=DIMENSION_MIN_FLOOR
    )

    base = {
        "hps": float(hps) if np.isfinite(hps) else np.nan,
        "hps_review": (
            float(hps_review) if np.isfinite(hps_review) else np.nan
        ),
        "hps_review_midpoint": float(REVIEW_MIDPOINT),
        "hps_review_margin": (
            float(hps_review - REVIEW_MIDPOINT)
            if np.isfinite(hps_review) else np.nan
        ),
        "critical_review_dimensions": ";".join(CRITICAL_REVIEW_DIMS),
        "critical_review_weights_json": json.dumps(
            CRITICAL_REVIEW_WEIGHTS, sort_keys=True
        ),
        "dimensions": dims,
        "dimension_min_floor": float(DIMENSION_MIN_FLOOR),
        "dimension_floor_failures": ";".join(dim_floor_failed),
        "evidence_validity_pass": bool(validity["passed"]),
        "evidence_factor_count_expected": int(
            validity["factor_count_expected"]
        ),
        "evidence_factor_count_observed_valid_numeric": int(
            validity["factor_count_observed_valid_numeric"]
        ),
        "evidence_missing": ";".join(validity["missing"]),
        "evidence_nonfinite": ";".join(validity["nonfinite"]),
        "evidence_nonpositive": ";".join(validity["nonpositive"]),
    }

    # Gate 1 — evidence validity across all 28 factors.
    if not validity["passed"]:
        failures = (
            validity["missing"]
            + validity["nonfinite"]
            + validity["nonpositive"]
        )
        return {
            **base,
            "decision_path": "EVIDENCE_VALIDITY_GATE",
            "initial_action": "AUTO_REJECT",
            "final_action": "REJECT",
            "status": "AUTO_REJECTED_EVIDENCE_VALIDITY",
            "reason": (
                "At least one of the 28 factor scores is missing, non-finite, "
                "or <=0: " + ";".join(failures)
            ),
        }

    # Gate 2 — every dimension must clear the 2.5/5 floor.
    if dim_floor_failed:
        return {
            **base,
            "decision_path": "DIMENSION_FLOOR",
            "initial_action": "AUTO_REJECT",
            "final_action": "REJECT",
            "status": "AUTO_REJECTED_DIMENSION_FLOOR",
            "reason": (
                f"At least one averaged dimension is below "
                f"{DIMENSION_MIN_FLOOR:.1f}/5: "
                + ";".join(
                    f"{dim}={dims[dim]:.3f}"
                    for dim in dim_floor_failed
                )
            ),
        }

    # Gate 3a — low global HPS.
    if hps < float(good_cut):
        return {
            **base,
            "decision_path": "LOW_HPS",
            "initial_action": "AUTO_REJECT",
            "final_action": "REJECT",
            "status": "AUTO_REJECTED_LOW_HPS",
            "reason": f"HPS {hps:.3f} < {good_cut:.3f}",
        }

    # Gate 3b — high global HPS.
    if hps >= float(high_cut):
        return {
            **base,
            "decision_path": "DIRECT_AUTO_ACCEPT",
            "initial_action": "AUTO_ACCEPT",
            "final_action": "ACCEPT",
            "status": "DIRECT_AUTO_ACCEPTED",
            "reason": (
                "All 28 factor scores valid; every dimension >= "
                f"{DIMENSION_MIN_FLOOR:.1f}; HPS {hps:.3f} >= "
                f"{high_cut:.3f}"
            ),
        }

    # Gate 4 — borderline global HPS; use the critical-dimension HPS_review.
    review_pass = bool(
        np.isfinite(hps_review)
        and float(hps_review) >= float(REVIEW_MIDPOINT)
    )

    if review_pass:
        return {
            **base,
            "decision_path": "HPS_REVIEW_BAND",
            "initial_action": "AUTOMATED_REVIEW",
            "final_action": "ACCEPT",
            "status": "ACCEPTED_AFTER_AUTOMATED_REVIEW",
            "reason": (
                f"HPS {hps:.3f} in [{good_cut:.3f}, {high_cut:.3f}); "
                f"HPS_review {hps_review:.3f} >= midpoint "
                f"{REVIEW_MIDPOINT:.3f}"
            ),
        }

    return {
        **base,
        "decision_path": "HPS_REVIEW_BAND",
        "initial_action": "AUTOMATED_REVIEW",
        "final_action": "REJECT",
        "status": "AUTO_REJECTED_HPS_REVIEW",
        "reason": (
            f"HPS {hps:.3f} in [{good_cut:.3f}, {high_cut:.3f}); "
            f"HPS_review {hps_review:.3f} < midpoint "
            f"{REVIEW_MIDPOINT:.3f}"
        ),
    }


def build_tadp_governance(
    client_ids: List[str],
    documentary_evidence: Dict[str, Dict[str, Dict[str, float]]],
    dq_scores: Dict[str, Dict[str, float]],
    run: int,
    evidence_seed: int,
    domain: str,
) -> pd.DataFrame:
    domain = str(domain).lower()
    rows = []

    for cid in client_ids:
        factors = {
            dim: dict(documentary_evidence[cid][dim])
            for dim in DOCUMENTARY_DIMS
        }
        factors["dim2"] = {
            name: float(dq_scores[cid][name])
            for name in FACTOR_NAMES["dim2"]
        }

        d = tadp_decision(factors, domain=domain)

        row = {
            "run": int(run),
            "domain": domain,
            "evidence_seed": int(evidence_seed),
            "client": str(cid),
            "hps": d["hps"],
            "hps_review": d["hps_review"],
            "hps_review_midpoint": d["hps_review_midpoint"],
            "hps_review_margin": d["hps_review_margin"],
            "critical_review_dimensions": d[
                "critical_review_dimensions"
            ],
            "critical_review_weights_json": d[
                "critical_review_weights_json"
            ],
            "evidence_validity_pass": bool(
                d["evidence_validity_pass"]
            ),
            "evidence_factor_count_expected": int(
                d["evidence_factor_count_expected"]
            ),
            "evidence_factor_count_observed_valid_numeric": int(
                d["evidence_factor_count_observed_valid_numeric"]
            ),
            "evidence_missing": d["evidence_missing"],
            "evidence_nonfinite": d["evidence_nonfinite"],
            "evidence_nonpositive": d["evidence_nonpositive"],
            "dimension_min_floor": float(d["dimension_min_floor"]),
            "dimension_floor_failures": d[
                "dimension_floor_failures"
            ],
            "decision_path": d["decision_path"],
            "initial_action": d["initial_action"],
            "final_action": d["final_action"],
            "status": d["status"],
            "reason": d["reason"],
        }

        for dim, value in d["dimensions"].items():
            row[f"{dim}_score_0_5"] = (
                float(value) if np.isfinite(value) else np.nan
            )

        rows.append(row)

    return pd.DataFrame(rows)


def accepted_tadp(governance_df: pd.DataFrame) -> List[str]:
    return governance_df.loc[
        governance_df["final_action"].eq("ACCEPT"), "client"
    ].astype(str).tolist()


def build_full_factor_evidence_table(
    client_ids: List[str],
    documentary_evidence: Dict[str, Dict[str, Dict[str, float]]],
    dq_scores: Dict[str, Dict[str, float]],
    dq_audit: pd.DataFrame,
    evidence_seed: int,
    domain: str,
) -> pd.DataFrame:
    """
    Full 28-factor evidence audit used by the final A1 policy.

    Policy fields are intentionally simple:
      - every factor must be present, finite, and >0;
      - dim1/dim2/dim5/dim6 are tagged as critical-review dimensions;
      - no factor-specific mandatory minima and no 14-factor review subset exist.
    """
    domain = str(domain).lower()

    dq_index = dq_audit.copy()
    dq_index["client"] = dq_index["client"].astype(str)
    dq_index = dq_index.set_index("client", drop=False)

    rows = []

    for cid in client_ids:
        cid = str(cid)

        for dim in DOCUMENTARY_DIMS:
            for factor in FACTOR_NAMES[dim]:
                score = float(documentary_evidence[cid][dim][factor])
                rows.append({
                    "client": cid,
                    "domain": domain,
                    "dimension": dim,
                    "dimension_name": DIMENSION_NAMES[dim],
                    "factor": factor,
                    "factor_source": "CONTROLLED_DOCUMENTARY_EVIDENCE",
                    "raw_measured_value": np.nan,
                    "rubric_score_0_5": score,
                    "rubric_descriptor": rubric_descriptor(dim, factor, score),
                    "factor_valid_for_gate": bool(
                        np.isfinite(score) and score > 0.0
                    ),
                    "is_critical_review_dimension": bool(
                        dim in CRITICAL_REVIEW_DIMS
                    ),
                    "evidence_seed": int(evidence_seed),
                })

        for factor in FACTOR_NAMES["dim2"]:
            score = float(dq_scores[cid][factor])
            raw_col = DQ_RAW_METRIC_BY_FACTOR[factor]
            raw_value = (
                float(dq_index.loc[cid, raw_col])
                if raw_col in dq_index.columns
                else np.nan
            )

            rows.append({
                "client": cid,
                "domain": domain,
                "dimension": "dim2",
                "dimension_name": DIMENSION_NAMES["dim2"],
                "factor": factor,
                "factor_source": "MACHINE_MEASURED_RAW_TRAIN_SHARD",
                "raw_measured_value": raw_value,
                "rubric_score_0_5": score,
                "rubric_descriptor": rubric_descriptor("dim2", factor, score),
                "factor_valid_for_gate": bool(
                    np.isfinite(score) and score > 0.0
                ),
                "is_critical_review_dimension": True,
                "evidence_seed": int(evidence_seed),
            })

    out = pd.DataFrame(rows)
    expected_rows = len(client_ids) * 28
    if len(out) != expected_rows:
        raise RuntimeError(
            f"Full evidence matrix should contain {expected_rows} rows; "
            f"found {len(out)}."
        )
    return out

# ======================================================================================
# GREAT EXPECTATIONS

# ======================================================================================
# GREAT EXPECTATIONS (GX CORE) — NATIVE VALIDATOR BASELINE
# ======================================================================================
# GX is used here in its native role: validate each client's TRAIN-only data against
# a predefined Expectation Suite and use the suite-level success flag as PASS/FAIL.
# There is NO ranking, NO forced-K selection, and NO TADP governance signal in this
# baseline. Great Expectations reports suite success=True only when all configured
# Expectations pass. The suite deliberately uses common technical checks only:
#   1) schema, 2) datatype consistency, 3) required-value ranges,
#   4) missingness, 5) duplicates / unique IDs, 6) label/domain validity,
#   7) structural integrity.
# Distribution checks are intentionally omitted because non-IID client distributions
# are expected in federated learning and should not by themselves constitute failure.
GX_CORE_VERSION = "1.23.0"
GX_VALUE_MOSTLY = 0.99
GX_NONNULL_REFERENCE_MIN = 0.80
GX_NONNULL_TOLERANCE = 0.15


def ensure_great_expectations():
    """Import pinned GX Core; install once in Colab if unavailable."""
    try:
        import great_expectations as gx
        return gx
    except ImportError:
        import subprocess
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            f"great_expectations=={GX_CORE_VERSION}",
        ])
        import great_expectations as gx
        return gx


def _gx_meta(category: str, name: str) -> Dict[str, Any]:
    return {
        "dq_category": str(category),
        "check_name": str(name),
        "baseline": "GX_NATIVE_VALIDATOR",
    }


def _gx_add(suite, specs: List[Dict[str, Any]], expectation, category: str, name: str):
    suite.add_expectation(expectation)
    specs.append({"category": str(category), "name": str(name)})


def _aggregate_nonnull_reference(
    client_frames: Dict[str, pd.DataFrame],
    columns: List[str],
) -> Dict[str, float]:
    total_rows = float(sum(len(df) for df in client_frames.values()))
    out = {}
    for c in columns:
        nonnull = sum(int(df[c].notna().sum()) for df in client_frames.values() if c in df.columns)
        out[c] = float(nonnull / max(1.0, total_rows))
    return out


def build_gx_healthcare_reference(client_frames: Dict[str, pd.DataFrame], preprocessor: Any) -> Dict[str, Any]:
    first = next(iter(client_frames.values()))

    # GX datatype validation is intentionally independent of TADP's model
    # preprocessing type inference.  The preprocessor labels a feature numeric
    # when >=95% of pooled TRAIN non-missing values are parseable; reusing that
    # inferred list inside GX and then requiring 99% parseability per client
    # creates an artificial contradiction.  GX therefore validates datatype
    # consistency only for fields whose numeric meaning is explicit in the
    # Diabetes data schema.
    known_numeric_fields = [
        "time_in_hospital",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses",
    ]
    gx_numeric_cols = [c for c in known_numeric_fields if c in first.columns]

    return {
        "original_columns": list(first.columns),
        "numeric_cols": gx_numeric_cols,
        "feature_cols": list(preprocessor.feature_cols),
        "nonnull_reference": _aggregate_nonnull_reference(
            client_frames, list(preprocessor.feature_cols)
        ),
    }


def build_gx_healthcare_validation_frame(df: pd.DataFrame, preprocessor: Any) -> pd.DataFrame:
    """GX validation view; raw TRAIN records remain the source of all checks."""
    out = df.copy()
    for c in preprocessor.numeric_cols:
        raw = df[c]
        parsed = pd.to_numeric(raw, errors="coerce")
        type_ok = raw.isna() | parsed.notna()
        out[c] = parsed.astype(float)
        out[f"__gx_type_ok__{c}"] = type_ok.astype(np.int8)
    return out


def build_gx_healthcare_suite(gx, reference: Dict[str, Any]):
    """Simple native GX technical-validation suite for the Diabetes TRAIN shards."""
    gxe = gx.expectations
    suite = gx.ExpectationSuite(name="tadp_healthcare_gx_native_suite")
    specs = []
    cols = reference["original_columns"]

    # 1) Schema.
    _gx_add(
        suite, specs,
        gxe.ExpectTableColumnsToMatchSet(
            column_set=cols,
            exact_match=False,
            severity="critical",
            meta=_gx_meta("Schema", "required_column_set"),
        ),
        "Schema", "required_column_set",
    )

    # 2) Datatype consistency for columns inferred as numeric from TRAIN only.
    for c in reference["numeric_cols"]:
        diag = f"__gx_type_ok__{c}"
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeInSet(
                column=diag,
                value_set=[1],
                mostly=GX_VALUE_MOSTLY,
                severity="critical",
                meta=_gx_meta("Datatype Consistency", f"{c}_numeric_parseability"),
            ),
            "Datatype Consistency", f"{c}_numeric_parseability",
        )

    # 3) Required-value ranges for known count/duration fields.
    nonnegative_fields = [
        "num_lab_procedures", "num_procedures", "num_medications",
        "number_outpatient", "number_emergency", "number_inpatient",
        "number_diagnoses",
    ]
    for c in nonnegative_fields:
        if c in cols:
            _gx_add(
                suite, specs,
                gxe.ExpectColumnValuesToBeBetween(
                    column=c, min_value=0.0, mostly=GX_VALUE_MOSTLY,
                    severity="critical",
                    meta=_gx_meta("Required Value Ranges", f"{c}_nonnegative"),
                ),
                "Required Value Ranges", f"{c}_nonnegative",
            )
    if "time_in_hospital" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeBetween(
                column="time_in_hospital", min_value=1.0, max_value=14.0,
                mostly=GX_VALUE_MOSTLY,
                severity="critical",
                meta=_gx_meta("Required Value Ranges", "time_in_hospital_range"),
            ),
            "Required Value Ranges", "time_in_hospital_range",
        )

    # 4) Missingness. Only columns that are substantially populated in the frozen
    # TRAIN reference are treated as required-enough for a missingness expectation.
    for c, global_nonnull in reference["nonnull_reference"].items():
        if c not in cols or float(global_nonnull) < GX_NONNULL_REFERENCE_MIN:
            continue
        minimum = max(0.70, float(global_nonnull) - GX_NONNULL_TOLERANCE)
        _gx_add(
            suite, specs,
            gxe.ExpectColumnProportionOfNonNullValuesToBeBetween(
                column=c, min_value=float(minimum), max_value=1.0,
                severity="warning",
                meta=_gx_meta("Missingness", f"{c}_nonnull"),
            ),
            "Missingness", f"{c}_nonnull",
        )

    # 5) Duplicates / unique identifiers.
    if "encounter_id" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeUnique(
                column="encounter_id",
                severity="critical",
                meta=_gx_meta("Duplicates / Unique IDs", "encounter_id_unique"),
            ),
            "Duplicates / Unique IDs", "encounter_id_unique",
        )

    # 6) Labels / domain validity.
    if "_target" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeInSet(
                column="_target", value_set=[0, 1, 2],
                severity="critical",
                meta=_gx_meta("Labels / Domain Validity", "target_domain"),
            ),
            "Labels / Domain Validity", "target_domain",
        )

    # 7) Structural integrity.
    if "_row_id" in cols:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeUnique(
                column="_row_id",
                severity="critical",
                meta=_gx_meta("Structural Integrity", "row_id_unique"),
            ),
            "Structural Integrity", "row_id_unique",
        )
    _gx_add(
        suite, specs,
        gxe.ExpectTableRowCountToBeBetween(
            min_value=100,
            severity="warning",
            meta=_gx_meta("Structural Integrity", "minimum_client_rows"),
        ),
        "Structural Integrity", "minimum_client_rows",
    )
    return suite, specs


def build_gx_cifar_metadata(X: np.ndarray, y: np.ndarray) -> pd.DataFrame:
    X = np.asarray(X)
    y = np.asarray(y).reshape(-1)
    if X.ndim != 4:
        raise RuntimeError(f"Expected CIFAR tensor [N,H,W,C], got shape={X.shape}")
    finite = np.isfinite(X.astype(np.float32)).reshape(len(X), -1).all(axis=1)
    flat = X.reshape(len(X), -1)
    return pd.DataFrame({
        "sample_id": np.arange(len(X), dtype=np.int64),
        "label": y.astype(np.int32),
        "height": np.full(len(X), X.shape[1], dtype=np.int32),
        "width": np.full(len(X), X.shape[2], dtype=np.int32),
        "channels": np.full(len(X), X.shape[3], dtype=np.int32),
        "dtype_ok": np.full(len(X), int(X.dtype == np.uint8), dtype=np.int8),
        "finite": finite.astype(np.int8),
        "pixel_min": flat.min(axis=1).astype(float),
        "pixel_max": flat.max(axis=1).astype(float),
    })


def build_gx_cifar_reference(client_raw: Dict[str, Tuple[np.ndarray, np.ndarray]]) -> Dict[str, Any]:
    # Native validator uses fixed technical constraints; no distribution reference needed.
    return {}


def build_gx_cifar_suite(gx, reference: Dict[str, Any]):
    """Simple native GX technical-validation suite for CIFAR-10 client metadata."""
    gxe = gx.expectations
    suite = gx.ExpectationSuite(name="tadp_cifar10_gx_native_suite")
    specs = []
    columns = [
        "sample_id", "label", "height", "width", "channels",
        "dtype_ok", "finite", "pixel_min", "pixel_max",
    ]

    _gx_add(
        suite, specs,
        gxe.ExpectTableColumnsToMatchSet(
            column_set=columns, exact_match=True,
            meta=_gx_meta("Schema", "image_metadata_schema"),
        ),
        "Schema", "image_metadata_schema",
    )
    for c in columns:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToNotBeNull(
                column=c,
                severity="warning",
                meta=_gx_meta("Missingness", f"{c}_not_null"),
            ),
            "Missingness", f"{c}_not_null",
        )
    for c, value in [("height", 32), ("width", 32), ("channels", 3), ("dtype_ok", 1), ("finite", 1)]:
        category = "Datatype Consistency" if c in {"dtype_ok", "finite"} else "Structural Integrity"
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeInSet(
                column=c, value_set=[value],
                meta=_gx_meta(category, f"{c}_constraint"),
            ),
            category, f"{c}_constraint",
        )
    for c in ["pixel_min", "pixel_max"]:
        _gx_add(
            suite, specs,
            gxe.ExpectColumnValuesToBeBetween(
                column=c, min_value=0.0, max_value=255.0,
                meta=_gx_meta("Required Value Ranges", f"{c}_valid_range"),
            ),
            "Required Value Ranges", f"{c}_valid_range",
        )
    _gx_add(
        suite, specs,
        gxe.ExpectColumnValuesToBeInSet(
            column="label", value_set=list(range(10)),
            meta=_gx_meta("Labels / Domain Validity", "label_domain"),
        ),
        "Labels / Domain Validity", "label_domain",
    )
    _gx_add(
        suite, specs,
        gxe.ExpectColumnValuesToBeUnique(
            column="sample_id",
            meta=_gx_meta("Duplicates / Unique IDs", "sample_id_unique"),
        ),
        "Duplicates / Unique IDs", "sample_id_unique",
    )
    _gx_add(
        suite, specs,
        gxe.ExpectTableRowCountToBeBetween(
            min_value=100,
            severity="warning",
            meta=_gx_meta("Structural Integrity", "minimum_client_images"),
        ),
        "Structural Integrity", "minimum_client_images",
    )
    return suite, specs


def ge_governance(
    client_data: Dict[str, Any],
    client_ids: List[str],
    accept_count: Optional[int] = None,  # ignored; retained only for call compatibility
    domain: str = "healthcare",
    preprocessor: Optional[Any] = None,
    dq_scores: Optional[Dict[str, Dict[str, float]]] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Run native GX suite validation and PASS only clients whose entire suite succeeds."""
    domain = str(domain).lower()
    gx = ensure_great_expectations()
    context = gx.get_context(mode="ephemeral")
    datasource = context.data_sources.add_pandas(name=f"tadp_gx_native_{domain}_datasource")
    asset = datasource.add_dataframe_asset(name=f"tadp_gx_native_{domain}_asset")
    batch_definition = asset.add_batch_definition_whole_dataframe(name="whole_client_train_partition")

    if domain == "healthcare":
        if preprocessor is None:
            raise ValueError("Healthcare GX native baseline requires TRAIN-only preprocessor metadata.")
        reference = build_gx_healthcare_reference(client_data, preprocessor)
        suite, specs = build_gx_healthcare_suite(gx, reference)
        make_frame = lambda cid: build_gx_healthcare_validation_frame(client_data[cid], preprocessor)
    elif domain == "cifar10":
        reference = build_gx_cifar_reference(client_data)
        suite, specs = build_gx_cifar_suite(gx, reference)
        def make_frame(cid):
            X, y = client_data[cid]
            return build_gx_cifar_metadata(X, y)
    else:
        raise ValueError(f"Unsupported GX domain: {domain!r}")

    summary_rows, detail_rows = [], []
    for cid in client_ids:
        frame = make_frame(cid)
        batch = batch_definition.get_batch(batch_parameters={"dataframe": frame})
        validation = batch.validate(suite)
        results = list(validation.results)
        if len(results) != len(specs):
            raise RuntimeError(
                f"GX result/spec mismatch for client {cid}: {len(results)} vs {len(specs)}"
            )

        passed = 0
        critical_failures = 0
        warning_failures = 0
        info_failures = 0
        category_totals, category_passed = {}, {}
        observed_names = []

        for result in results:
            success = bool(result.success)
            passed += int(success)
            cfg = result.expectation_config
            meta = getattr(cfg, "meta", None) or {}
            name = str(meta.get("check_name", getattr(cfg, "type", "GX_EXPECTATION")))
            cat = str(meta.get("dq_category", "Technical Validation"))

            sev_obj = getattr(cfg, "severity", None)
            sev = getattr(sev_obj, "value", sev_obj)
            sev = str(sev if sev is not None else "critical").lower()
            if "." in sev:
                sev = sev.split(".")[-1]
            if sev not in {"critical", "warning", "info"}:
                sev = "critical"

            if not success:
                if sev == "critical":
                    critical_failures += 1
                elif sev == "warning":
                    warning_failures += 1
                else:
                    info_failures += 1

            observed_names.append(name)
            category_totals[cat] = category_totals.get(cat, 0) + 1
            category_passed[cat] = category_passed.get(cat, 0) + int(success)

            detail_rows.append({
                "client": str(cid),
                "domain": domain,
                "gx_version": GX_CORE_VERSION,
                "expectation_name": name,
                "dq_category": cat,
                "severity": sev,
                "success": success,
            })

        expected_names = sorted(str(x["name"]) for x in specs)
        if sorted(observed_names) != expected_names:
            raise RuntimeError(
                f"GX expectation identity mismatch for client {cid}: "
                f"expected={expected_names}, observed={sorted(observed_names)}"
            )

        total = len(specs)
        stats = getattr(validation, "statistics", {}) or {}
        suite_success = bool(validation.success)

        # GX-native severity-aware operational gate.
        # GX itself exposes the maximum failed severity for a Validation Result.
        # We use that native result rather than reconstructing the gate from a
        # custom ranking.  A failed Expectation execution is also treated by GX
        # as CRITICAL.
        max_failed_severity_obj = validation.get_max_severity_failure()
        if max_failed_severity_obj is None:
            max_failed_severity = "none"
        else:
            max_failed_severity = str(
                getattr(max_failed_severity_obj, "value", max_failed_severity_obj)
            ).lower()
            if "." in max_failed_severity:
                max_failed_severity = max_failed_severity.split(".")[-1]

        operational_valid = (max_failed_severity != "critical")

        row = {
            "client": str(cid),
            "gx_version": GX_CORE_VERSION,
            "gx_native_suite_success": suite_success,
            "gx_operational_valid": bool(operational_valid),
            "gx_critical_failures": int(critical_failures),
            "gx_warning_failures": int(warning_failures),
            "gx_info_failures": int(info_failures),
            "gx_max_failed_severity": max_failed_severity,
            "ge_expectations_passed": int(stats.get("successful_expectations", passed)),
            "ge_expectations_total": int(stats.get("evaluated_expectations", total)),
            "ge_pass_rate": float(
                stats.get("success_percent", 100.0 * passed / max(1, total))
            ) / 100.0,
            "ge_native_validation_class": (
                "GX_SUITE_PASS"
                if suite_success
                else (
                    "GX_WARNING_ONLY"
                    if operational_valid
                    else "GX_CRITICAL_FAILURE"
                )
            ),
            "ge_final_action": "ACCEPT" if operational_valid else "REJECT",
            "ge_policy": (
                "GX Core severity-aware validation; ACCEPT requires zero critical "
                "Expectation failures; warning/info failures are reported but do not "
                "exclude; no ranking and no forced-K selection"
            ),
        }

        for cat in sorted(category_totals):
            safe = re.sub(r"[^a-z0-9]+", "_", cat.lower()).strip("_")
            row[f"ge_{safe}_passed"] = int(category_passed.get(cat, 0))
            row[f"ge_{safe}_total"] = int(category_totals[cat])

        summary_rows.append(row)

    return pd.DataFrame(summary_rows), pd.DataFrame(detail_rows)


def score_lower_is_better(value: float, cuts: List[float]) -> float:
    """
    cuts = [best_upper, score4_upper, score3_upper, score2_upper, score1_upper]
    value <= cuts[0] => 5; ... value <= cuts[4] => 1; else 0.
    """
    v = float(value)
    for score, upper in zip([5, 4, 3, 2, 1], cuts):
        if v <= float(upper):
            return float(score)
    return 0.0


def score_higher_is_better(value: float, cuts: List[float]) -> float:
    """
    cuts = [score5_lower, score4_lower, score3_lower, score2_lower, score1_lower]
    """
    v = float(value)
    for score, lower in zip([5, 4, 3, 2, 1], cuts):
        if v >= float(lower):
            return float(score)
    return 0.0


def js_divergence(p, q, eps=1e-12) -> float:
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / max(eps, p.sum())
    q = q / max(eps, q.sum())
    m = 0.5 * (p + q)
    kl_pm = np.sum(np.where(p > 0, p * np.log((p + eps) / (m + eps)), 0.0))
    kl_qm = np.sum(np.where(q > 0, q * np.log((q + eps) / (m + eps)), 0.0))
    return float(0.5 * (kl_pm + kl_qm))


# ======================================================================================
# MODEL / METRICS / FL TRAINING
# ======================================================================================

def class_weight_dict(y: np.ndarray) -> Dict[int, float]:
    y = np.asarray(y, dtype=np.int32)
    classes = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {int(c): float(w) for c, w in zip(classes, weights)}


def model_parameter_bytes(model: keras.Model) -> int:
    return int(sum(np.asarray(w).nbytes for w in model.get_weights()))


def evaluate_model(
    model: keras.Model, X: np.ndarray, y: np.ndarray, n_classes: int
) -> Dict[str, float]:
    p = model.predict(X, batch_size=512, verbose=0)
    pred = np.argmax(p, axis=1)
    out = {
        "accuracy": float(accuracy_score(y, pred)),
        "precision_macro": float(
            precision_score(y, pred, average="macro", zero_division=0)
        ),
        "recall_macro": float(
            recall_score(y, pred, average="macro", zero_division=0)
        ),
        "f1_macro": float(
            f1_score(y, pred, average="macro", zero_division=0)
        ),
    }
    try:
        out["roc_auc_ovr_macro"] = float(
            roc_auc_score(y, p, multi_class="ovr", average="macro")
        )
    except Exception:
        out["roc_auc_ovr_macro"] = float("nan")
    return out



ROUND_PROGRESS_MONITOR_MAX_SAMPLES = 4096

# ----------------------------------------------------------------------
# OPERATIONAL ACCOUNTING ASSUMPTIONS
# ----------------------------------------------------------------------
# Runtime and byte counts are measured/derived directly by the experiment.
# Energy, carbon, and monetary values are reference-scenario estimates.
#
# IMPORTANT:
# - REFERENCE_AVG_COMPUTE_POWER_W is NOT a measured Colab power draw.
# - REFERENCE_GRID_CO2_KG_PER_KWH is a fixed reference intensity.
# - Electricity and communication prices are scenario assumptions.
# These constants are exported with the results so the estimates can be
# recalculated without retraining.
REFERENCE_AVG_COMPUTE_POWER_W = 12.0
REFERENCE_GRID_CO2_KG_PER_KWH = 0.445
REFERENCE_ELECTRICITY_USD_PER_KWH = 0.20
REFERENCE_COMMUNICATION_USD_PER_MB = 0.005

# Progress-only alias; kept identical to the reference energy model.
ROUND_PROGRESS_POWER_W = REFERENCE_AVG_COMPUTE_POWER_W


def build_train_monitor_subset(
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    max_samples: int = ROUND_PROGRESS_MONITOR_MAX_SAMPLES,
    seed: int = 99117,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Build a deterministic lightweight monitoring subset from TRAIN client arrays only.

    This subset is used solely for console progress after FL rounds. It never affects
    preprocessing, governance, client selection, optimizer budgets, checkpoint
    decisions, or final reporting. The held-out TEST set remains final-evaluation-only.
    """
    rng = np.random.default_rng(int(seed))
    client_ids = sorted(client_arrays.keys())
    sizes = {cid: int(len(client_arrays[cid][1])) for cid in client_ids}
    total = int(sum(sizes.values()))
    target = int(min(max_samples, total))

    # Proportional allocation across clients, then distribute rounding remainder.
    raw = {cid: target * sizes[cid] / max(1, total) for cid in client_ids}
    take = {cid: min(sizes[cid], int(np.floor(raw[cid]))) for cid in client_ids}

    while sum(take.values()) < target:
        candidates = [c for c in client_ids if take[c] < sizes[c]]
        if not candidates:
            break
        cid = max(candidates, key=lambda c: (raw[c] - take[c], sizes[c], c))
        take[cid] += 1

    xs, ys = [], []
    for cid in client_ids:
        n_take = int(take[cid])
        if n_take <= 0:
            continue
        Xc, yc = client_arrays[cid]
        if n_take >= len(yc):
            idx = np.arange(len(yc))
        else:
            idx = rng.choice(len(yc), size=n_take, replace=False)
        xs.append(np.asarray(Xc[idx]))
        ys.append(np.asarray(yc[idx], dtype=np.int32))

    Xmon = np.concatenate(xs, axis=0)
    ymon = np.concatenate(ys, axis=0)

    # Deterministic shuffle so monitoring batches do not follow client order.
    order = rng.permutation(len(ymon))
    return Xmon[order], ymon[order]


def _fmt_metric(x: float) -> str:
    return "nan" if not np.isfinite(float(x)) else f"{float(x):.4f}"


def print_round_progress(
    progress_context: Optional[Dict[str, Any]],
    round_idx: int,
    round_total: int,
    selected_ids: List[str],
    round_steps: int,
    cumulative_steps: int,
    monitor_metrics: Dict[str, float],
    cumulative_runtime_s: float,
    cumulative_communication_mb: float,
    ram_start_mb: float,
    ram_peak_mb: float,
    extra: str = "",
) -> None:
    """Compact, human-readable progress block after each federated round."""
    ctx = progress_context or {}
    scenario = str(ctx.get("scenario", "Federated scenario"))
    run_idx = int(ctx.get("run_idx", 0))
    run_total = int(ctx.get("run_total", 0))
    scenario_idx = int(ctx.get("scenario_idx", 0))
    scenario_total = int(ctx.get("scenario_total", 0))
    overall_idx = int(ctx.get("overall_idx", 0))
    overall_total = int(ctx.get("overall_total", 0))

    energy_wh = float(
        ROUND_PROGRESS_POWER_W * float(cumulative_runtime_s) / 3600.0
    )
    ram_delta = max(0.0, float(ram_peak_mb) - float(ram_start_mb))

    print("\n" + "-" * 112)
    print(
        f"PROGRESS | overall configuration {overall_idx}/{overall_total} | "
        f"run {run_idx}/{run_total} | scenario {scenario_idx}/{scenario_total}"
    )
    print(f"SCENARIO | {scenario}")
    print(
        f"ROUND    | {round_idx}/{round_total} | "
        f"selected={len(selected_ids)} [{','.join(map(str, selected_ids))}] | "
        f"steps={round_steps} | cumulative_steps={cumulative_steps}"
    )
    if extra:
        print(f"DETAIL   | {extra}")
    print(
        "TRAIN-MONITOR (diagnostic only; TEST untouched) | "
        f"Accuracy={_fmt_metric(monitor_metrics.get('accuracy', np.nan))} | "
        f"F1={_fmt_metric(monitor_metrics.get('f1_macro', np.nan))} | "
        f"AUC={_fmt_metric(monitor_metrics.get('roc_auc_ovr_macro', np.nan))} | "
        f"Precision={_fmt_metric(monitor_metrics.get('precision_macro', np.nan))} | "
        f"Recall={_fmt_metric(monitor_metrics.get('recall_macro', np.nan))}"
    )
    print(
        f"CUMULATIVE OPERATIONAL | runtime={cumulative_runtime_s:.2f}s | "
        f"energy_ref≈{energy_wh:.5f}Wh | payload={cumulative_communication_mb:.3f}MB | "
        f"RAM peak={ram_peak_mb:.1f}MB | RAM Δ={ram_delta:.1f}MB"
    )
    print("-" * 112)


def train_exact_steps(
    model: keras.Model,
    X: np.ndarray,
    y: np.ndarray,
    steps: int,
    batch_size: int,
    seed: int,
    class_weights: Optional[Dict[int, float]] = None,
    prox_reference: Optional[List[np.ndarray]] = None,
    prox_mu: float = 0.0,
):
    """
    Exact mini-batch update count. Used for step-parity audits.
    """
    steps = int(max(1, steps))
    rng = np.random.default_rng(int(seed))
    n = len(y)
    if n == 0:
        raise RuntimeError("Cannot train on an empty dataset.")

    loss_fn = keras.losses.SparseCategoricalCrossentropy(
        reduction=keras.losses.Reduction.NONE
    )

    order = rng.permutation(n)
    cursor = 0

    prox_tensors = None
    if prox_reference is not None and prox_mu > 0:
        prox_tensors = [tf.convert_to_tensor(w) for w in prox_reference]

    for _ in range(steps):
        if cursor + batch_size > n:
            order = rng.permutation(n)
            cursor = 0

        idx = order[cursor:cursor + batch_size]
        cursor += batch_size

        xb = tf.convert_to_tensor(np.asarray(X[idx]), dtype=tf.float32)
        yb_np = np.asarray(y[idx], dtype=np.int32)
        yb = tf.convert_to_tensor(yb_np, dtype=tf.int32)

        with tf.GradientTape() as tape:
            probs = model(xb, training=True)
            per_loss = loss_fn(yb, probs)

            if class_weights:
                sw = np.array(
                    [class_weights.get(int(v), 1.0) for v in yb_np],
                    dtype=np.float32,
                )
                sw_t = tf.convert_to_tensor(sw)
                data_loss = tf.reduce_sum(per_loss * sw_t) / tf.reduce_sum(sw_t)
            else:
                data_loss = tf.reduce_mean(per_loss)

            loss = data_loss

            if prox_tensors is not None:
                prox = tf.constant(0.0, dtype=tf.float32)
                for var, ref in zip(model.trainable_variables, prox_tensors):
                    prox += tf.reduce_sum(tf.square(var - tf.cast(ref, var.dtype)))
                loss = loss + 0.5 * float(prox_mu) * prox

        grads = tape.gradient(loss, model.trainable_variables)
        model.optimizer.apply_gradients(zip(grads, model.trainable_variables))


def natural_steps(n_records: int, batch_size: int, local_epochs: int = 1) -> int:
    return int(max(1, math.ceil(int(n_records) / int(batch_size)) * int(local_epochs)))


def allocate_exact_step_budget(
    selected: List[str],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    target_total: int,
    batch_size: int,
    local_epochs: int = 1,
) -> Dict[str, int]:
    natural = {
        cid: natural_steps(len(client_arrays[cid][1]), batch_size, local_epochs)
        for cid in selected
    }
    total_nat = max(1, sum(natural.values()))
    raw = {cid: target_total * natural[cid] / total_nat for cid in selected}
    alloc = {cid: max(1, int(math.floor(raw[cid]))) for cid in selected}

    # Adjust to exact target.
    while sum(alloc.values()) < target_total:
        cid = max(selected, key=lambda c: raw[c] - alloc[c])
        alloc[cid] += 1
    while sum(alloc.values()) > target_total:
        candidates = [c for c in selected if alloc[c] > 1]
        if not candidates:
            break
        cid = min(candidates, key=lambda c: raw[c] - alloc[c])
        alloc[cid] -= 1

    if sum(alloc.values()) != int(target_total):
        raise RuntimeError("Exact step-budget allocation failed.")
    return alloc


def aggregate_weights(
    local_weights: List[List[np.ndarray]],
    sample_sizes: List[int],
    equal_weight: bool = False,
) -> List[np.ndarray]:
    if not local_weights:
        raise RuntimeError("No local weights to aggregate.")
    if equal_weight:
        alpha = np.ones(len(local_weights), dtype=float) / len(local_weights)
    else:
        sizes = np.asarray(sample_sizes, dtype=float)
        alpha = sizes / sizes.sum()

    out = []
    for layer_idx in range(len(local_weights[0])):
        x = sum(alpha[j] * np.asarray(local_weights[j][layer_idx])
                for j in range(len(local_weights)))
        out.append(np.asarray(x))
    return out


def federated_train(
    build_model_fn,
    initial_weights: List[np.ndarray],
    selected_per_round: List[List[str]],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int,
    batch_size: int,
    local_epochs: int,
    class_weights: Dict[int, float],
    run_seed: int,
    exact_step_maps: Optional[List[Dict[str, int]]] = None,
    equal_weight: bool = False,
    fedprox_mu: float = 0.0,
    train_monitor: Optional[Tuple[np.ndarray, np.ndarray]] = None,
    progress_context: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    monitor = RAMMonitor().start()
    start = time.perf_counter()

    global_model = build_model_fn()
    global_model.set_weights([np.array(w, copy=True) for w in initial_weights])

    round_rows = []
    total_steps = 0
    total_selected = 0
    monitor_eval_s = 0.0
    cumulative_comm_raw_b = 0
    param_b = model_parameter_bytes(global_model)

    for r, selected in enumerate(selected_per_round, start=1):
        global_weights = [np.array(w, copy=True) for w in global_model.get_weights()]
        local_weights = []
        local_sizes = []
        round_steps = 0

        if exact_step_maps is None:
            step_map = {
                cid: natural_steps(
                    len(client_arrays[cid][1]), batch_size, local_epochs
                )
                for cid in selected
            }
        else:
            step_map = exact_step_maps[r - 1]

        for j, cid in enumerate(selected):
            Xc, yc = client_arrays[cid]
            local_model = build_model_fn()
            local_model.set_weights(global_weights)
            train_exact_steps(
                local_model, Xc, yc,
                steps=int(step_map[cid]),
                batch_size=batch_size,
                seed=int(run_seed + 1000 * r + 17 * j),
                class_weights=class_weights,
                prox_reference=global_weights if fedprox_mu > 0 else None,
                prox_mu=float(fedprox_mu),
            )
            local_weights.append(
                [np.array(w, copy=True) for w in local_model.get_weights()]
            )
            local_sizes.append(len(yc))
            round_steps += int(step_map[cid])

            del local_model
            tf.keras.backend.clear_session()

        agg = aggregate_weights(
            local_weights, local_sizes, equal_weight=equal_weight
        )
        global_model = build_model_fn()
        global_model.set_weights(agg)

        total_steps += round_steps
        total_selected += len(selected)

        # Exact application-level model payload through this round:
        # one global-model download + one model-update upload per selected client.
        # No arbitrary network/protocol overhead multiplier is applied.
        cumulative_comm_raw_b += int(len(selected)) * 2 * int(param_b)
        cumulative_comm_mb = float(cumulative_comm_raw_b / 1_000_000.0)

        # Measure training runtime BEFORE this round's diagnostic evaluation.
        # Previous diagnostic-evaluation time is subtracted so progress printing
        # does not inflate the experiment's runtime metric.
        cumulative_runtime_s = float(
            max(0.0, (time.perf_counter() - start) - monitor_eval_s)
        )

        if train_monitor is not None:
            Xmon, ymon = train_monitor
            t_mon = time.perf_counter()
            monitor_metrics = evaluate_model(global_model, Xmon, ymon, n_classes)
            monitor_eval_s += float(time.perf_counter() - t_mon)
        else:
            monitor_metrics = {
                "accuracy": float("nan"),
                "precision_macro": float("nan"),
                "recall_macro": float("nan"),
                "f1_macro": float("nan"),
                "roc_auc_ovr_macro": float("nan"),
            }

        live_peak = float(monitor.peak_mb)
        round_row = {
            "round": r,
            "selected_clients": len(selected),
            "selected_ids": ";".join(selected),
            "optimizer_steps": int(round_steps),
            "cumulative_optimizer_steps": int(total_steps),
            "train_monitor_accuracy": float(monitor_metrics["accuracy"]),
            "train_monitor_precision_macro": float(monitor_metrics["precision_macro"]),
            "train_monitor_recall_macro": float(monitor_metrics["recall_macro"]),
            "train_monitor_f1_macro": float(monitor_metrics["f1_macro"]),
            "train_monitor_auc_ovr_macro": float(monitor_metrics["roc_auc_ovr_macro"]),
            "cumulative_runtime_s": float(cumulative_runtime_s),
            "cumulative_energy_wh_est": float(
                ROUND_PROGRESS_POWER_W * cumulative_runtime_s / 3600.0
            ),
            "cumulative_communication_mb": float(cumulative_comm_mb),
            "ram_peak_mb_live": float(live_peak),
            "ram_delta_mb_live": float(max(0.0, live_peak - monitor.start_mb)),
        }
        round_rows.append(round_row)

        print_round_progress(
            progress_context=progress_context,
            round_idx=r,
            round_total=len(selected_per_round),
            selected_ids=list(selected),
            round_steps=int(round_steps),
            cumulative_steps=int(total_steps),
            monitor_metrics=monitor_metrics,
            cumulative_runtime_s=cumulative_runtime_s,
            cumulative_communication_mb=cumulative_comm_mb,
            ram_start_mb=float(monitor.start_mb),
            ram_peak_mb=live_peak,
        )

    runtime = float(max(0.0, (time.perf_counter() - start) - monitor_eval_s))
    metrics = evaluate_model(global_model, X_test, y_test, n_classes)
    ram = monitor.stop()

    communication_b = int(cumulative_comm_raw_b)

    return {
        "model": global_model,
        "metrics": metrics,
        "runtime_s": float(runtime),
        "total_optimizer_steps": int(total_steps),
        "communication_payload_bytes": int(communication_b),
        "communication_mb": float(communication_b / 1_000_000.0),
        "communication_mib": float(communication_b / (1024 ** 2)),
        "round_audit": round_rows,
        "participants_mean_per_round": float(
            total_selected / max(1, len(round_rows))
        ),
        **ram,
    }




def local_training_loss(model: keras.Model, X: np.ndarray, y: np.ndarray, batch_size: int = 512) -> float:
    """Mean sparse cross-entropy on client TRAIN data only; used by Power-of-Choice."""
    p = model.predict(X, batch_size=batch_size, verbose=0)
    y = np.asarray(y, dtype=np.int32)
    idx = np.arange(len(y))
    probs = np.clip(p[idx, y], 1e-12, 1.0)
    return float(-np.mean(np.log(probs)))


def federated_train_power_of_choice(
    build_model_fn,
    initial_weights: List[np.ndarray],
    candidate_clients: List[str],
    select_k: int,
    target_steps_per_round: int,
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int,
    batch_size: int,
    local_epochs: int,
    class_weights: Dict[int, float],
    run_seed: int,
    candidate_multiplier: int = 2,
    train_monitor: Optional[Tuple[np.ndarray, np.ndarray]] = None,
    progress_context: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """
    Power-of-Choice baseline: each round samples d candidates and selects the K
    clients with largest current local TRAIN loss. K, rounds, initialisation, and
    total optimizer steps per round are matched to TADP. No TEST information is used.
    """
    monitor = RAMMonitor().start()
    start = time.perf_counter()
    global_model = build_model_fn()
    global_model.set_weights([np.array(w, copy=True) for w in initial_weights])
    rng = np.random.default_rng(int(run_seed) + 880000)
    round_rows = []
    total_steps = 0
    total_selected = 0
    communication_b = 0
    monitor_eval_s = 0.0

    all_candidates = list(candidate_clients)
    if int(select_k) < 1 or int(select_k) > len(all_candidates):
        raise RuntimeError("Invalid Power-of-Choice K.")

    for r in range(1, NUM_ROUNDS_FL + 1):
        d = min(len(all_candidates), max(int(select_k), int(candidate_multiplier) * int(select_k)))
        if d == len(all_candidates):
            candidate_pool = list(all_candidates)
        else:
            # Canonical pow-d samples candidate clients without replacement
            # according to p_k, the client's fraction of total TRAIN data.
            sizes = np.asarray(
                [len(client_arrays[c][1]) for c in all_candidates], dtype=float
            )
            probs = sizes / sizes.sum()
            candidate_pool = rng.choice(
                all_candidates, size=d, replace=False, p=probs
            ).tolist()

        losses = []
        for cid in candidate_pool:
            Xc, yc = client_arrays[cid]
            losses.append((str(cid), local_training_loss(global_model, Xc, yc)))
        losses.sort(key=lambda x: (-x[1], x[0]))
        selected = [cid for cid, _ in losses[:int(select_k)]]
        step_map = allocate_exact_step_budget(
            selected, client_arrays, int(target_steps_per_round), batch_size, local_epochs
        )

        global_weights = [np.array(w, copy=True) for w in global_model.get_weights()]
        local_weights, local_sizes = [], []
        for j, cid in enumerate(selected):
            Xc, yc = client_arrays[cid]
            local_model = build_model_fn()
            local_model.set_weights(global_weights)
            train_exact_steps(
                local_model, Xc, yc,
                steps=int(step_map[cid]), batch_size=batch_size,
                seed=int(run_seed + 1000 * r + 17 * j),
                class_weights=class_weights,
            )
            local_weights.append([np.array(w, copy=True) for w in local_model.get_weights()])
            local_sizes.append(len(yc))
            del local_model
            tf.keras.backend.clear_session()

        agg = aggregate_weights(local_weights, local_sizes, equal_weight=False)
        global_model = build_model_fn()
        global_model.set_weights(agg)
        round_steps = int(sum(step_map.values()))
        total_steps += round_steps
        total_selected += len(selected)
        param_b = model_parameter_bytes(global_model)
        # Application-level PoC payload accounting:
        #   (i) each candidate receives the current global model,
        #  (ii) each candidate returns one float64 local-loss score,
        # (iii) each selected client returns one model update.
        # This intentionally excludes transport/TLS/header overhead because the
        # experiment does not emulate a physical network stack.
        loss_score_bytes = int(len(candidate_pool)) * 8
        communication_b += (
            int(len(candidate_pool)) * int(param_b)
            + int(len(selected)) * int(param_b)
            + loss_score_bytes
        )
        cumulative_comm_mb = float(communication_b / 1_000_000.0)
        cumulative_runtime_s = float(
            max(0.0, (time.perf_counter() - start) - monitor_eval_s)
        )

        if train_monitor is not None:
            Xmon, ymon = train_monitor
            t_mon = time.perf_counter()
            monitor_metrics = evaluate_model(global_model, Xmon, ymon, n_classes)
            monitor_eval_s += float(time.perf_counter() - t_mon)
        else:
            monitor_metrics = {
                "accuracy": float("nan"),
                "precision_macro": float("nan"),
                "recall_macro": float("nan"),
                "f1_macro": float("nan"),
                "roc_auc_ovr_macro": float("nan"),
            }

        live_peak = float(monitor.peak_mb)
        round_rows.append({
            "round": r,
            "candidate_count": len(candidate_pool),
            "candidate_ids": ";".join(candidate_pool),
            "selected_clients": len(selected),
            "selected_ids": ";".join(selected),
            "optimizer_steps": round_steps,
            "cumulative_optimizer_steps": int(total_steps),
            "local_losses": json.dumps({cid: loss for cid, loss in losses}, sort_keys=True),
            "train_monitor_accuracy": float(monitor_metrics["accuracy"]),
            "train_monitor_precision_macro": float(monitor_metrics["precision_macro"]),
            "train_monitor_recall_macro": float(monitor_metrics["recall_macro"]),
            "train_monitor_f1_macro": float(monitor_metrics["f1_macro"]),
            "train_monitor_auc_ovr_macro": float(monitor_metrics["roc_auc_ovr_macro"]),
            "cumulative_runtime_s": float(cumulative_runtime_s),
            "cumulative_energy_wh_est": float(
                ROUND_PROGRESS_POWER_W * cumulative_runtime_s / 3600.0
            ),
            "cumulative_communication_mb": float(cumulative_comm_mb),
            "ram_peak_mb_live": float(live_peak),
            "ram_delta_mb_live": float(max(0.0, live_peak - monitor.start_mb)),
        })

        loss_preview = ", ".join(
            f"{cid}:{loss:.3f}" for cid, loss in losses[:min(5, len(losses))]
        )
        print_round_progress(
            progress_context=progress_context,
            round_idx=r,
            round_total=NUM_ROUNDS_FL,
            selected_ids=list(selected),
            round_steps=int(round_steps),
            cumulative_steps=int(total_steps),
            monitor_metrics=monitor_metrics,
            cumulative_runtime_s=cumulative_runtime_s,
            cumulative_communication_mb=cumulative_comm_mb,
            ram_start_mb=float(monitor.start_mb),
            ram_peak_mb=live_peak,
            extra=f"PoC candidates={len(candidate_pool)} | highest TRAIN losses: {loss_preview}",
        )

    runtime = float(max(0.0, (time.perf_counter() - start) - monitor_eval_s))
    metrics = evaluate_model(global_model, X_test, y_test, n_classes)
    ram = monitor.stop()
    communication_b = int(communication_b)
    return {
        "model": global_model,
        "metrics": metrics,
        "runtime_s": float(runtime),
        "total_optimizer_steps": int(total_steps),
        "communication_payload_bytes": int(communication_b),
        "communication_mb": float(communication_b / 1_000_000.0),
        "communication_mib": float(communication_b / (1024 ** 2)),
        "round_audit": round_rows,
        "participants_mean_per_round": float(total_selected / max(1, NUM_ROUNDS_FL)),
        **ram,
    }


def centralized_train(
    build_model_fn,
    initial_weights: List[np.ndarray],
    selected_clients: List[str],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int,
    batch_size: int,
    exact_steps: int,
    class_weights: Dict[int, float],
    seed: int,
) -> Dict[str, Any]:
    if not selected_clients:
        raise RuntimeError("Centralized scenario has no TRAIN clients.")

    monitor = RAMMonitor().start()
    start = time.perf_counter()

    # Pool ACCEPTED TRAIN partitions only. TEST is not present here.
    X = np.concatenate([client_arrays[c][0] for c in selected_clients], axis=0)
    y = np.concatenate([client_arrays[c][1] for c in selected_clients], axis=0)

    model = build_model_fn()
    model.set_weights([np.array(w, copy=True) for w in initial_weights])
    train_exact_steps(
        model, X, y,
        steps=int(exact_steps),
        batch_size=batch_size,
        seed=int(seed),
        class_weights=class_weights,
    )

    runtime = time.perf_counter() - start
    metrics = evaluate_model(model, X_test, y_test, n_classes)
    ram = monitor.stop()

    # The experiment is a single-machine simulation, so this is NOT measured
    # network traffic. It is the exact logical one-time model-ready TRAIN payload
    # represented by the selected dense feature arrays and integer labels.
    central_payload_b = int(
        sum(
            np.asarray(client_arrays[c][0]).nbytes
            + np.asarray(client_arrays[c][1]).nbytes
            for c in selected_clients
        )
    )

    return {
        "model": model,
        "metrics": metrics,
        "runtime_s": float(runtime),
        "total_optimizer_steps": int(exact_steps),
        "communication_payload_bytes": int(central_payload_b),
        "communication_mb": float(central_payload_b / 1_000_000.0),
        "communication_mib": float(central_payload_b / (1024 ** 2)),
        "participants_mean_per_round": float(len(selected_clients)),
        **ram,
    }


def result_row(
    run: int,
    seed: int,
    scenario: str,
    result: Dict[str, Any],
    initial_hash: str,
    power_w: float = REFERENCE_AVG_COMPUTE_POWER_W,
) -> Dict[str, Any]:
    row = {
        "run": int(run),
        "seed": int(seed),
        "scenario": str(scenario),
        **result["metrics"],
        "runtime_s": float(result["runtime_s"]),
        "optimizer_steps": int(result["total_optimizer_steps"]),
        "communication_payload_bytes": int(result.get("communication_payload_bytes", 0)),
        "communication_mb": float(result.get("communication_mb", 0.0)),
        "communication_mib": float(result.get("communication_mib", 0.0)),
        "participants": float(result.get("participants_mean_per_round", 0.0)),
        "ram_start_mb": float(result.get("ram_start_mb", 0.0)),
        "ram_end_mb": float(result.get("ram_end_mb", 0.0)),
        "ram_peak_mb": float(result.get("ram_peak_mb", 0.0)),
        "ram_delta_mb": float(result.get("ram_delta_mb", 0.0)),
        "ram_mb": float(result.get("ram_mb", result.get("ram_peak_mb", 0.0))),
        "initial_weights_sha256": initial_hash,
    }
    # Runtime is measured. Energy/carbon/cost are reference-scenario estimates.
    # energy_wh_per_watt is assumption-free and allows recomputation for any
    # alternative average-power estimate without rerunning model training.
    row["energy_wh_per_watt"] = float(row["runtime_s"] / 3600.0)
    row["reference_avg_compute_power_w"] = float(power_w)
    row["energy_wh"] = float(power_w * row["energy_wh_per_watt"])
    row["energy_kwh"] = float(row["energy_wh"] / 1000.0)
    row["reference_grid_co2_kg_per_kwh"] = float(
        REFERENCE_GRID_CO2_KG_PER_KWH
    )
    row["co2_kg"] = float(
        row["energy_kwh"] * REFERENCE_GRID_CO2_KG_PER_KWH
    )
    row["reference_electricity_usd_per_kwh"] = float(
        REFERENCE_ELECTRICITY_USD_PER_KWH
    )
    row["reference_communication_usd_per_mb"] = float(
        REFERENCE_COMMUNICATION_USD_PER_MB
    )
    row["energy_cost_usd"] = float(
        row["energy_kwh"] * REFERENCE_ELECTRICITY_USD_PER_KWH
    )
    row["communication_cost_usd"] = float(
        row["communication_mb"] * REFERENCE_COMMUNICATION_USD_PER_MB
    )
    row["total_estimated_cost_usd"] = float(
        row["energy_cost_usd"] + row["communication_cost_usd"]
    )
    row["energy_is_estimated"] = True
    row["communication_is_application_payload_not_network_capture"] = True
    return row


# ======================================================================================
# LEAKAGE AUDIT
# ======================================================================================

def write_leakage_audit(
    out_dir: Path,
    train_ids,
    test_ids,
    client_train_ids: Dict[str, np.ndarray],
    extra: Optional[Dict[str, Any]] = None,
):
    train_set = set(map(str, train_ids))
    test_set = set(map(str, test_ids))
    overlap = train_set & test_set

    client_union = set()
    duplicates_across_clients = 0
    for cid, ids in client_train_ids.items():
        s = set(map(str, ids))
        duplicates_across_clients += len(client_union & s)
        client_union |= s

    test_in_clients = len(test_set & client_union)
    missing_train = len(train_set - client_union)
    extra_client_rows = len(client_union - train_set)

    row = {
        "train_test_overlap": len(overlap),
        "test_rows_in_any_client": test_in_clients,
        "train_rows_missing_from_clients": missing_train,
        "client_rows_not_in_global_train": extra_client_rows,
        "duplicate_train_rows_across_clients": duplicates_across_clients,
        "pass": (
            len(overlap) == 0
            and test_in_clients == 0
            and missing_train == 0
            and extra_client_rows == 0
            and duplicates_across_clients == 0
        ),
    }
    if extra:
        row.update(extra)

    pd.DataFrame([row]).to_csv(
        Path(out_dir) / "leakage_audit.csv", index=False
    )

    if not bool(row["pass"]):
        raise RuntimeError(f"FAIL-CLOSED leakage audit failed: {row}")
    return row


# ======================================================================================
# DIABETES 130-US — LEAKAGE-SAFE DATA PREPARATION
# ======================================================================================

TARGET = "readmitted"
ID_COLUMNS = ["encounter_id", "patient_nbr"]


def locate_diabetes_csv() -> str:
    """
    Reuse the dataset already uploaded by the separate Colab upload cell.

    Preferred cached path:
        ./data_cache/diabetes_130US.csv

    This function intentionally NEVER opens an upload dialog.
    """
    candidates = [
        os.environ.get("DIABETES_CSV", ""),
        "./data_cache/diabetes_130US.csv",
        "/content/data_cache/diabetes_130US.csv",
        "./diabetes_130US.csv",
        "/content/diabetes_130US.csv",
        "/content/drive/MyDrive/diabetes_130US.csv",
    ]

    for p in candidates:
        if p and os.path.isfile(p):
            resolved = os.path.abspath(p)
            print(f"✅ Reusing existing Diabetes dataset: {resolved}")
            return p

    raise FileNotFoundError(
        "diabetes_130US.csv was not found. Run the SEPARATE dataset-upload cell first. "
        "That cell should cache the file as ./data_cache/diabetes_130US.csv. "
        "The Experiment A1 code intentionally does not request another upload."
    )


def load_diabetes(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df = df.replace("?", np.nan)

    if TARGET not in df.columns:
        raise RuntimeError(f"Missing target column: {TARGET}")

    valid_target = {"NO": 0, ">30": 1, "<30": 2}
    df = df[df[TARGET].isin(valid_target)].copy()
    df["_target"] = df[TARGET].map(valid_target).astype(np.int32)
    df["_row_id"] = np.arange(len(df), dtype=np.int64)

    return df


def global_patient_grouped_split(
    df: pd.DataFrame, seed: int, test_fraction: float = 0.20
):
    """
    Patient-grouped and stratified whenever patient_nbr is available.
    The split happens before any client partitioning or data-dependent preprocessing.
    """
    if "patient_nbr" in df.columns:
        splitter = StratifiedGroupKFold(
            n_splits=5, shuffle=True, random_state=int(seed)
        )
        train_idx, test_idx = next(
            splitter.split(
                np.zeros(len(df)),
                y=df["_target"].to_numpy(),
                groups=df["patient_nbr"].astype(str).to_numpy(),
            )
        )
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        patient_overlap = len(
            set(train_df["patient_nbr"].astype(str))
            & set(test_df["patient_nbr"].astype(str))
        )
        if patient_overlap != 0:
            raise RuntimeError("Patient leakage detected across TRAIN/TEST.")
    else:
        train_df, test_df = train_test_split(
            df, test_size=float(test_fraction),
            stratify=df["_target"], random_state=int(seed)
        )
        patient_overlap = 0

    return train_df.reset_index(drop=True), test_df.reset_index(drop=True), patient_overlap


def dirichlet_partition_dataframe(
    train_df: pd.DataFrame,
    n_clients: int,
    alpha: float,
    seed: int,
    min_client_records: int = 100,
) -> Dict[str, pd.DataFrame]:
    y = train_df["_target"].to_numpy()
    client_ids = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")[:n_clients]

    for attempt in range(100):
        rng = np.random.default_rng(int(seed + attempt))
        buckets = [[] for _ in range(n_clients)]

        for cls in sorted(np.unique(y)):
            idx = np.where(y == cls)[0]
            rng.shuffle(idx)
            props = rng.dirichlet(np.full(n_clients, float(alpha)))
            cuts = (np.cumsum(props)[:-1] * len(idx)).astype(int)
            splits = np.split(idx, cuts)
            for k, s in enumerate(splits):
                buckets[k].extend(s.tolist())

        sizes = [len(b) for b in buckets]
        if min(sizes) >= int(min_client_records):
            out = {}
            for cid, idxs in zip(client_ids, buckets):
                out[cid] = train_df.iloc[np.array(idxs, dtype=int)].copy()
            return out

    raise RuntimeError("Could not obtain a valid Dirichlet client partition.")



@dataclass
class TabularDQSchema:
    """
    Governance-only TRAIN schema used to measure Data Quality before admission.

    This object does NOT contain model scaling parameters, categorical model
    vocabularies, class weights, or transformed arrays.
    """
    feature_cols: List[str]
    numeric_cols: List[str]
    categorical_cols: List[str]


def infer_train_only_dq_schema(
    client_frames: Dict[str, pd.DataFrame]
) -> TabularDQSchema:
    """
    Infer only the raw TRAIN feature/type schema required by the eight DQ checks.
    This is a governance-stage operation, not model preprocessing.
    """
    any_df = next(iter(client_frames.values()))
    feature_cols = [
        c for c in any_df.columns
        if c not in {TARGET, "_target", "_row_id", *ID_COLUMNS}
    ]

    numeric_cols = []
    categorical_cols = []
    for c in feature_cols:
        total_nonmissing = 0
        numeric_valid = 0
        for df in client_frames.values():
            raw = df[c]
            nm = raw.notna()
            total_nonmissing += int(nm.sum())
            if nm.any():
                numeric_valid += int(
                    pd.to_numeric(
                        raw[nm], errors="coerce"
                    ).notna().sum()
                )
        ratio = numeric_valid / max(1, total_nonmissing)
        if ratio >= 0.95:
            numeric_cols.append(c)
        else:
            categorical_cols.append(c)

    return TabularDQSchema(
        feature_cols=feature_cols,
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
    )


@dataclass
class TabularPreprocessor:
    feature_cols: List[str]
    numeric_cols: List[str]
    categorical_cols: List[str]
    mean: Dict[str, float]
    std: Dict[str, float]
    categories: Dict[str, List[str]]
    encoder: Any

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        pieces = []

        if self.numeric_cols:
            x_num = []
            for c in self.numeric_cols:
                s = pd.to_numeric(df[c], errors="coerce").astype(float)
                a = s.fillna(self.mean[c]).to_numpy(dtype=np.float32)
                a = (a - self.mean[c]) / self.std[c]
                x_num.append(a[:, None])
            pieces.append(np.concatenate(x_num, axis=1).astype(np.float32))

        if self.categorical_cols:
            cat = pd.DataFrame({
                c: df[c].astype("string").fillna("__MISSING__").astype(str)
                for c in self.categorical_cols
            })
            x_cat = self.encoder.transform(cat)
            pieces.append(np.asarray(x_cat, dtype=np.float32))

        if not pieces:
            raise RuntimeError("No predictor columns remained.")
        return np.concatenate(pieces, axis=1).astype(np.float32)


def fit_federated_train_only_preprocessor(
    client_frames: Dict[str, pd.DataFrame]
) -> TabularPreprocessor:
    """
    No raw TRAIN pooling is used to ESTIMATE numeric parameters.
    Numeric mean/std comes from aggregated local count/sum/sum-of-squares.
    Categorical vocabulary comes from union of local TRAIN category sets.
    """
    any_df = next(iter(client_frames.values()))
    feature_cols = [
        c for c in any_df.columns
        if c not in {TARGET, "_target", "_row_id", *ID_COLUMNS}
    ]

    # Infer expected type from TRAIN only.
    numeric_cols = []
    categorical_cols = []
    for c in feature_cols:
        total_nonmissing = 0
        numeric_valid = 0
        for df in client_frames.values():
            raw = df[c]
            nm = raw.notna()
            total_nonmissing += int(nm.sum())
            if nm.any():
                numeric_valid += int(
                    pd.to_numeric(raw[nm], errors="coerce").notna().sum()
                )
        ratio = numeric_valid / max(1, total_nonmissing)
        if ratio >= 0.95:
            numeric_cols.append(c)
        else:
            categorical_cols.append(c)

    mean = {}
    std = {}
    for c in numeric_cols:
        count = 0
        sum_ = 0.0
        sumsq = 0.0
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            count += len(a)
            sum_ += float(a.sum())
            sumsq += float(np.square(a).sum())
        mu = sum_ / max(1, count)
        var = max(1e-12, sumsq / max(1, count) - mu * mu)
        mean[c] = float(mu)
        std[c] = float(math.sqrt(var))

    categories = {}
    for c in categorical_cols:
        values = set()
        for df in client_frames.values():
            s = df[c].astype("string").fillna("__MISSING__").astype(str)
            values.update(s.unique().tolist())
        categories[c] = sorted(values)

    encoder = OneHotEncoder(
        categories=[categories[c] for c in categorical_cols],
        handle_unknown="ignore",
        sparse_output=False,
        dtype=np.float32,
    )

    # Fit only metadata-shaped dummy rows; the vocabulary is already frozen from TRAIN.
    if categorical_cols:
        max_len = max(len(categories[c]) for c in categorical_cols)
        dummy = {}
        for c in categorical_cols:
            vals = categories[c]
            dummy[c] = [vals[i % len(vals)] for i in range(max_len)]
        encoder.fit(pd.DataFrame(dummy))

    return TabularPreprocessor(
        feature_cols=feature_cols,
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
        mean=mean,
        std=std,
        categories=categories,
        encoder=encoder,
    )


def build_tabular_reference(
    client_frames: Dict[str, pd.DataFrame],
    preprocessor: Any,
) -> Dict[str, Any]:
    """
    TRAIN-only DQ reference built from client-local summaries only.

    Numerical reference histograms are constructed by combining local
    min/max summaries and then summing client-local histogram counts.
    Categorical reference support is the union of local TRAIN category sets.
    Raw client records are not concatenated to construct the reference.
    """
    ref = {"num_hist": {}, "cat_values": {}}

    chosen_num = preprocessor.numeric_cols[:12]
    for c in chosen_num:
        local_min, local_max = [], []
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            if len(a):
                local_min.append(float(np.min(a)))
                local_max.append(float(np.max(a)))
        if not local_min:
            continue
        gmin, gmax = float(min(local_min)), float(max(local_max))
        if gmax <= gmin:
            edges = np.array([gmin - 1e-6, gmax + 1e-6], dtype=float)
        else:
            edges = np.linspace(gmin, gmax, 11, dtype=float)
        global_hist = np.zeros(len(edges)-1, dtype=np.float64)
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            if len(a):
                h, _ = np.histogram(a, bins=edges)
                global_hist += h.astype(np.float64)
        ref["num_hist"][c] = {
            "edges": edges.tolist(),
            "hist": global_hist.tolist(),
            "construction": "aggregated_client_local_histograms_only",
        }

    for c in preprocessor.categorical_cols:
        values = set()
        for df in client_frames.values():
            values.update(
                df[c].astype("string").fillna("__MISSING__")
                .astype(str).unique().tolist()
            )
        ref["cat_values"][c] = sorted(values)
    return ref

def tabular_dq_scores(
    df: pd.DataFrame,
    preprocessor: Any,
    reference: Dict[str, Any],
) -> Tuple[Dict[str, float], Dict[str, float]]:
    """
    Eight machine-measured TRAIN-only DQ factors for the healthcare experiment.
    """
    feature_df = df[preprocessor.feature_cols]

    # 1) Completeness.
    missing_fraction = float(feature_df.isna().mean().mean())
    completeness = score_lower_is_better(
        missing_fraction,
        [0.01, 0.05, 0.10, 0.20, 0.50],
    )

    # 2) Duplication rate.
    duplicate_fraction = float(feature_df.duplicated().mean())
    duplication = score_lower_is_better(
        duplicate_fraction,
        [0.01, 0.02, 0.05, 0.10, 0.20],
    )

    # 3) Value validity / error rate.
    bad = 0
    observed = 0

    for c in preprocessor.numeric_cols:
        raw = df[c]
        nm = raw.notna()
        observed += int(nm.sum())

        if nm.any():
            conv = pd.to_numeric(
                raw[nm], errors="coerce"
            ).to_numpy(dtype=float)
            bad += int(np.sum(~np.isfinite(conv)))

    for c in preprocessor.categorical_cols:
        raw = df[c]
        nm = raw.notna()
        observed += int(nm.sum())

        if nm.any():
            bad += int(
                np.sum(raw[nm].astype(str).str.strip().eq(""))
            )

    error_fraction = float(bad / max(1, observed))
    value_validity_error_rate = score_lower_is_better(
        error_fraction,
        [0.01, 0.02, 0.05, 0.10, 0.15],
    )

    # 4) Type consistency.
    type_bad = 0
    type_obs = 0

    for c in preprocessor.numeric_cols:
        raw = df[c]
        nm = raw.notna()
        type_obs += int(nm.sum())

        if nm.any():
            type_bad += int(
                pd.to_numeric(
                    raw[nm], errors="coerce"
                ).isna().sum()
            )

    type_inconsistency = float(type_bad / max(1, type_obs))
    type_consistency = score_lower_is_better(
        type_inconsistency,
        [0.01, 0.02, 0.05, 0.10, 0.20],
    )

    # 5) Label integrity.
    invalid_label_fraction = float(
        (~df["_target"].isin([0, 1, 2])).mean()
    )
    label_integrity = score_lower_is_better(
        invalid_label_fraction,
        [0.001, 0.01, 0.02, 0.05, 0.10],
    )

    # 6) Feature-distribution consistency — TRAIN-only reference.
    jsds = []

    for c, spec in reference["num_hist"].items():
        a = pd.to_numeric(
            df[c], errors="coerce"
        ).to_numpy(dtype=float)
        a = a[np.isfinite(a)]

        if len(a):
            hist, _ = np.histogram(
                a,
                bins=np.array(spec["edges"], dtype=float),
            )
            jsds.append(
                js_divergence(
                    hist,
                    np.array(spec["hist"], dtype=float),
                )
            )

    max_jsd = float(max(jsds)) if jsds else 0.0
    distribution_consistency = score_lower_is_better(
        max_jsd,
        [0.01, 0.025, 0.05, 0.10, 0.20],
    )

    # 7) Feature/category coverage.
    coverage_vals = []

    for c, ref_vals in reference["cat_values"].items():
        ref_set = set(ref_vals)

        if ref_set:
            client_set = set(
                df[c]
                .astype("string")
                .fillna("__MISSING__")
                .astype(str)
                .unique()
            )
            coverage_vals.append(
                len(client_set & ref_set) / len(ref_set)
            )

    mean_coverage = (
        float(np.mean(coverage_vals))
        if coverage_vals else 1.0
    )
    feature_coverage = score_higher_is_better(
        mean_coverage,
        [0.90, 0.825, 0.75, 0.65, 0.50],
    )

    # 8) Structural / constraint integrity.
    # Uses only TRAIN records. It checks:
    #   - key presence / encounter uniqueness;
    #   - non-negative count-like clinical fields;
    #   - finite numeric values where a numeric value is expected.
    n = len(df)
    record_violation = np.zeros(n, dtype=bool)

    if "encounter_id" not in df.columns:
        record_violation[:] = True
    else:
        encounter = df["encounter_id"]
        record_violation |= encounter.isna().to_numpy()
        record_violation |= encounter.duplicated(keep=False).to_numpy()

    nonnegative_fields = [
        "time_in_hospital",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses",
    ]

    for c in nonnegative_fields:
        if c in df.columns:
            a = pd.to_numeric(
                df[c], errors="coerce"
            ).to_numpy(dtype=float)
            bad_c = (~np.isfinite(a)) | (a < 0)
            record_violation |= bad_c

    structural_violation_fraction = float(
        np.mean(record_violation)
    ) if n else 1.0

    structural_integrity = score_lower_is_better(
        structural_violation_fraction,
        [0.001, 0.01, 0.02, 0.05, 0.10],
    )

    scores = {
        "completeness": completeness,
        "duplication_rate": duplication,
        "value_validity_error_rate": value_validity_error_rate,
        "type_consistency": type_consistency,
        "label_integrity": label_integrity,
        "feature_distribution_consistency": distribution_consistency,
        "feature_category_coverage": feature_coverage,
        "structural_constraint_integrity": structural_integrity,
    }

    raw = {
        "missing_fraction": missing_fraction,
        "duplicate_fraction": duplicate_fraction,
        "error_fraction": error_fraction,
        "type_inconsistency_fraction": type_inconsistency,
        "invalid_label_fraction": invalid_label_fraction,
        "max_jsd": max_jsd,
        "mean_category_coverage": mean_coverage,
        "structural_violation_fraction": structural_violation_fraction,
    }

    return scores, raw

def prepare_diabetes_governance_stage(
    csv_path: str,
    split_seed: int,
    partition_seed: int,
    n_clients: int = 10,
    alpha: float = 1.0,
):
    """
    OPTION A ordering.

    1) Create GLOBAL held-out TEST.
    2) Partition only GLOBAL TRAIN into raw client shards.
    3) Measure DQ and execute governance on raw TRAIN shards.
    4) DO NOT fit model scaler, categorical vocabulary, class weights, or create
       transformed model arrays here.

    Model preprocessing is fitted later, only after a scenario's participating
    cohort has been determined.
    """
    t_prepare0 = time.perf_counter()

    raw = load_diabetes(csv_path)
    train_df, test_df, patient_overlap = global_patient_grouped_split(
        raw, split_seed
    )

    clients = dirichlet_partition_dataframe(
        train_df,
        n_clients=n_clients,
        alpha=alpha,
        seed=partition_seed,
    )
    t_before_dq = time.perf_counter()
    client_ids = list(clients.keys())

    client_train_ids = {
        cid: df["_row_id"].astype(str).to_numpy()
        for cid, df in clients.items()
    }

    # Governance-only raw TRAIN schema/reference. These objects are used solely
    # to measure DQ before admission; they are not model preprocessing artifacts.
    dq_schema = infer_train_only_dq_schema(clients)
    dq_reference = build_tabular_reference(clients, dq_schema)

    dq_scores = {}
    dq_raw_rows = []
    for cid, df in clients.items():
        scores, raw_metrics = tabular_dq_scores(
            df, dq_schema, dq_reference
        )
        dq_scores[cid] = scores
        dq_raw_rows.append({
            "client": cid,
            **scores,
            **raw_metrics,
        })

    t_after_dq = time.perf_counter()

    meta = {
        "raw_rows": int(len(raw)),
        "train_rows": int(len(train_df)),
        "test_rows": int(len(test_df)),
        "patient_overlap": int(patient_overlap),
        "n_clients": int(n_clients),
        "governance_feature_count": int(len(dq_schema.feature_cols)),
        "governance_numeric_feature_count": int(
            len(dq_schema.numeric_cols)
        ),
        "governance_categorical_feature_count": int(
            len(dq_schema.categorical_cols)
        ),
        "model_preprocessing_fitted_before_governance": False,
        "class_weights_computed_before_governance": False,
        "raw_load_split_partition_s": float(t_before_dq - t_prepare0),
        "dq_measurement_s": float(t_after_dq - t_before_dq),
        "governance_input_preparation_total_s": float(t_after_dq - t_prepare0),
    }

    return {
        "raw": raw,
        "train_df": train_df,
        "test_df": test_df,
        "clients_raw": clients,
        "client_ids": client_ids,
        "dq_scores": dq_scores,
        "dq_audit": pd.DataFrame(dq_raw_rows),
        "meta": meta,
        "client_train_ids": client_train_ids,
        "global_train_ids": train_df["_row_id"].astype(str).to_numpy(),
        "global_test_ids": test_df["_row_id"].astype(str).to_numpy(),
        "dq_schema": dq_schema,
        "dq_reference": dq_reference,
    }


def preprocessor_fingerprint(pre: TabularPreprocessor) -> str:
    payload = {
        "feature_cols": list(pre.feature_cols),
        "numeric_cols": list(pre.numeric_cols),
        "categorical_cols": list(pre.categorical_cols),
        "mean": {k: float(v) for k, v in pre.mean.items()},
        "std": {k: float(v) for k, v in pre.std.items()},
        "categories": {
            k: list(map(str, v)) for k, v in pre.categories.items()
        },
    }
    canonical = json.dumps(
        payload, sort_keys=True, separators=(",", ":")
    ).encode("utf-8")
    return hashlib.sha256(canonical).hexdigest()


def build_post_selection_training_view(
    clients_raw: Dict[str, pd.DataFrame],
    test_df: pd.DataFrame,
    participating_clients: List[str],
    view_name: str,
) -> Dict[str, Any]:
    """
    Fit model preprocessing ONLY from the participating TRAIN cohort.

    This function is called after the participation decision is frozen.
    Rejected/unselected clients do not contribute numeric scaling statistics,
    categorical vocabulary, class weights, or transformed training records.
    """
    participating_clients = list(map(str, participating_clients))
    if not participating_clients:
        raise RuntimeError(
            f"Training view {view_name!r} has no participating clients."
        )

    selected_frames = {
        cid: clients_raw[cid]
        for cid in participating_clients
    }

    pre = fit_federated_train_only_preprocessor(selected_frames)

    client_arrays = {}
    for cid in participating_clients:
        df = clients_raw[cid]
        X = pre.transform(df)
        y = df["_target"].to_numpy(dtype=np.int32)
        client_arrays[cid] = (X, y)

    X_test = pre.transform(test_df)
    y_test = test_df["_target"].to_numpy(dtype=np.int32)

    y_train = np.concatenate(
        [client_arrays[cid][1] for cid in participating_clients]
    )
    cw = class_weight_dict(y_train)

    fp = preprocessor_fingerprint(pre)

    return {
        "view_name": str(view_name),
        "participating_clients": participating_clients,
        "participating_client_count": int(len(participating_clients)),
        "training_rows": int(
            sum(len(clients_raw[cid]) for cid in participating_clients)
        ),
        "preprocessor": pre,
        "preprocessor_sha256": fp,
        "client_arrays": client_arrays,
        "X_test": X_test,
        "y_test": y_test,
        "class_weights": cw,
        "input_dim": int(X_test.shape[1]),
        "numeric_feature_count": int(len(pre.numeric_cols)),
        "categorical_feature_count": int(len(pre.categorical_cols)),
        "categorical_encoded_width": int(
            X_test.shape[1] - len(pre.numeric_cols)
        ),
    }


def build_training_view_cache(
    clients_raw: Dict[str, pd.DataFrame],
    test_df: pd.DataFrame,
    cohorts: Dict[str, List[str]],
) -> Tuple[Dict[str, Dict[str, Any]], pd.DataFrame]:
    """
    Build one post-selection preprocessing view per UNIQUE participant cohort.
    Scenarios sharing exactly the same cohort share the same fitted transform.
    """
    cache = {}
    scenario_views = {}
    audit_rows = []

    for scenario, cohort in cohorts.items():
        key = tuple(sorted(map(str, cohort)))
        if key not in cache:
            cache[key] = build_post_selection_training_view(
                clients_raw=clients_raw,
                test_df=test_df,
                participating_clients=list(key),
                view_name=";".join(key),
            )
        view = cache[key]
        scenario_views[scenario] = view
        audit_rows.append({
            "scenario": str(scenario),
            "participating_clients": ";".join(view["participating_clients"]),
            "participating_client_count": int(
                view["participating_client_count"]
            ),
            "training_rows": int(view["training_rows"]),
            "input_dim": int(view["input_dim"]),
            "numeric_feature_count": int(
                view["numeric_feature_count"]
            ),
            "categorical_feature_count": int(
                view["categorical_feature_count"]
            ),
            "categorical_encoded_width": int(
                view["categorical_encoded_width"]
            ),
            "preprocessor_sha256": str(
                view["preprocessor_sha256"]
            ),
            "class_weights_json": json.dumps(
                {
                    str(k): float(v)
                    for k, v in view["class_weights"].items()
                },
                sort_keys=True,
            ),
            "preprocessing_fit_after_selection": True,
            "rejected_or_unselected_clients_in_preprocessing": False,
        })

    return scenario_views, pd.DataFrame(audit_rows)


def build_diabetes_model(input_dim: int, lr: float = 1e-3) -> keras.Model:


    inp = keras.Input(shape=(int(input_dim),), dtype=tf.float32)
    x = layers.Dense(128, activation="relu")(inp)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(3, activation="softmax", dtype=tf.float32)(x)
    model = keras.Model(inp, out)
    model.optimizer = keras.optimizers.Adam(learning_rate=float(lr))
    return model



# ======================================================================================
# FULL EXPERIMENT-A REPORTING, CHECKPOINTING, LEDGER, AND STATISTICS
# ======================================================================================
from datetime import datetime, timezone
import re

# =============================================================================
# MAIN REPEATED TRAINING SET
#
# Eight distinct benchmark configurations are repeated over all five training
# one-seed boundary/stress condition and is not pooled with the headline n=5
# statistics. GX Core is executed only as a raw-TRAIN technical validator and
# is not retrained as a headline predictive scenario in this A1 notebook.
# =============================================================================
MANUSCRIPT_SCENARIOS = [
    "Naïve Centralized",
    "TADP Centralized",
    "Vanilla FedAvg",
    "FedProx",
    "Random-K",
    "DQ-only Federated",
    "Power-of-Choice",
    "TADP Federated",
]
assert len(MANUSCRIPT_SCENARIOS) == 8




def select_dq_only_clients(
    dq_scores: Dict[str, Dict[str, float]],
    k: int,
) -> Tuple[List[str], pd.DataFrame]:
    """
    Matched data-quality-aware client-selection baseline.

    Each client is scored using ONLY the eight machine-measured Data Quality
    and Health factors that form TADP dimension 2. The DQ-only score is the
    arithmetic mean of those eight factor scores. Clients are ranked by that
    score and the top-k are selected, where k is the realized number of
    contributors admitted by TADP.

    Therefore DQ-only differs from TADP only in the selection signal while the
    matched comparison controls K, FL rounds, and realized optimizer-step
    budget. Great Expectations remains a separate independent validator and is
    not forced to this K.
    """
    rows = []
    for cid, scores in dq_scores.items():
        vals = np.asarray(
            [float(scores[f]) for f in FACTOR_NAMES["dim2"]],
            dtype=float,
        )

        valid = bool(
            len(vals) == len(FACTOR_NAMES["dim2"])
            and np.isfinite(vals).all()
            and np.all(vals > 0.0)
        )

        rows.append({
            "client": str(cid),
            "dq_only_score_0_5": float(np.mean(vals)) if valid else -np.inf,
            "dq_factor_count": int(len(vals)),
            "dq_evidence_valid": valid,
        })

    audit = pd.DataFrame(rows).sort_values(
        ["dq_only_score_0_5", "client"],
        ascending=[False, True],
    ).reset_index(drop=True)

    if int(k) < 1 or int(k) > len(audit):
        raise RuntimeError(
            f"Invalid matched DQ-only K={k}; available clients={len(audit)}."
        )

    valid_count = int(audit["dq_evidence_valid"].sum())
    if valid_count < int(k):
        raise RuntimeError(
            f"DQ-only cannot form a matched top-{k} cohort: "
            f"only {valid_count} clients have valid DQ evidence."
        )

    audit["dq_only_rank"] = np.arange(1, len(audit) + 1)
    selected = audit.head(int(k))["client"].astype(str).tolist()
    audit["dq_only_selected"] = audit["client"].isin(selected)
    audit["matched_target_k"] = int(k)

    return selected, audit


def print_banner(title: str, width: int = 108):
    print("\n" + "=" * width)
    print(title)
    print("=" * width)


def client_partition_table(clients_raw):
    rows = []
    for cid, df in clients_raw.items():
        counts = df["_target"].value_counts().to_dict()
        rows.append({
            "client": cid,
            "records": int(len(df)),
            "class_0_NO": int(counts.get(0, 0)),
            "class_1_GT30": int(counts.get(1, 0)),
            "class_2_LT30": int(counts.get(2, 0)),
        })
    return pd.DataFrame(rows)


def evidence_assignment_summary(
    evidence_df: pd.DataFrame
) -> pd.DataFrame:
    """Summarize the frozen controlled documentary-evidence assignment."""
    required = {
        "client",
        "bundle_id",
        "evidence_profile",
        "scenario_role",
        "profile_variant",
        "factor",
        "rubric_score_0_5",
        "factor_valid_for_gate",
        "evidence_seed",
    }
    missing = sorted(required - set(evidence_df.columns))
    if missing:
        raise RuntimeError(
            "Controlled documentary-evidence table is missing required "
            f"column(s): {missing}. Available columns: "
            f"{sorted(evidence_df.columns.tolist())}"
        )

    work = evidence_df.copy()
    work["factor_valid_for_gate"] = work[
        "factor_valid_for_gate"
    ].astype(bool)

    summary = (
        work
        .groupby(["client", "bundle_id"], as_index=False)
        .agg(
            evidence_profile=("evidence_profile", "first"),
            scenario_role=("scenario_role", "first"),
            profile_variant=("profile_variant", "first"),
            documentary_factor_count=("factor", "count"),
            documentary_valid_factor_count=("factor_valid_for_gate", "sum"),
            documentary_mean_score=("rubric_score_0_5", "mean"),
            documentary_min_score=("rubric_score_0_5", "min"),
            documentary_max_score=("rubric_score_0_5", "max"),
            evidence_seed=("evidence_seed", "first"),
        )
        .sort_values("client")
        .reset_index(drop=True)
    )

    expected_documentary_factors = sum(
        len(FACTOR_NAMES[d]) for d in DOCUMENTARY_DIMS
    )
    if not summary["documentary_factor_count"].eq(
        expected_documentary_factors
    ).all():
        raise RuntimeError(
            f"Expected {expected_documentary_factors} documentary factors "
            "per client."
        )

    summary["documentary_validity_fraction"] = (
        summary["documentary_valid_factor_count"]
        / summary["documentary_factor_count"].clip(lower=1)
    )
    return summary

def print_governance_details(
    gov: pd.DataFrame,
    ge: pd.DataFrame,
    dq: pd.DataFrame,
    tadp_clients: List[str],
    ge_clients: List[str],
    domain: str,
):
    print_banner("DOMAIN ADMISSION POLICY — 28-FACTOR VALIDITY + HPS + CRITICAL-DIMENSION REVIEW")
    print(
        "Evidence-validity gate: all 28 factor scores must be present, finite, "
        "and strictly >0."
    )
    print(
        f"Dimension floor: every averaged dimension must be >= "
        f"{DIMENSION_MIN_FLOOR:.1f}/5."
    )
    print(
        f"Main HPS: <{GOOD_CUT:.1f}=Reject; "
        f"[{GOOD_CUT:.1f},{HIGH_CUT:.1f})=Automated Review; "
        f">={HIGH_CUT:.1f}=Direct Accept."
    )
    print(
        "Automated Review critical dimensions: "
        + ", ".join(
            f"{d}={DIMENSION_NAMES[d]}" for d in CRITICAL_REVIEW_DIMS
        )
    )
    print(
        "Renormalized critical weights: "
        + json.dumps(CRITICAL_REVIEW_WEIGHTS, sort_keys=True)
    )
    print(
        f"HPS_review >= {REVIEW_MIDPOINT:.2f}/5 -> Accept after Review; "
        "otherwise Reject."
    )
    print(
        "Human reviewer role: verify documentary evidence only; "
        "the server performs scoring and admission automatically."
    )

    print_banner("RAW TRAIN-SHARD DATA-QUALITY FACTORS — 8 FACTORS")
    dq_cols = [
        "client",
        "completeness",
        "duplication_rate",
        "value_validity_error_rate",
        "type_consistency",
        "label_integrity",
        "feature_distribution_consistency",
        "feature_category_coverage",
        "structural_constraint_integrity",
    ]
    print(dq[dq_cols].to_string(index=False))

    print_banner("FROZEN TADP GOVERNANCE — FINAL A1 POLICY")
    display_cols = [
        "client",
        "evidence_validity_pass",
        "evidence_nonpositive",
        "hps",
        "hps_review",
        "hps_review_midpoint",
        "hps_review_margin",
        "dimension_floor_failures",
        "dim1_score_0_5",
        "dim2_score_0_5",
        "dim3_score_0_5",
        "dim4_score_0_5",
        "dim5_score_0_5",
        "dim6_score_0_5",
        "decision_path",
        "initial_action",
        "final_action",
        "status",
        "reason",
    ]
    print(gov[display_cols].to_string(index=False))

    print("\nFinal TADP A1 decision order:")
    print("  1) Any of 28 factor scores missing/non-finite/<=0 -> AUTO-REJECT")
    print(
        f"  2) Any averaged dimension < {DIMENSION_MIN_FLOOR:.1f}/5 "
        "-> AUTO-REJECT"
    )
    print(f"  3) HPS < {GOOD_CUT:.1f} -> AUTO-REJECT")
    print(f"  4) HPS >= {HIGH_CUT:.1f} -> DIRECT AUTO-ACCEPT")
    print(
        f"  5) {GOOD_CUT:.1f} <= HPS < {HIGH_CUT:.1f} "
        "-> AUTOMATED REVIEW"
    )
    print(
        f"       HPS_review >= {REVIEW_MIDPOINT:.2f}/5 "
        "-> ACCEPT AFTER REVIEW"
    )
    print("       otherwise -> AUTO-REJECT")

    print(
        f"\nFrozen TADP cohort: "
        f"{len(tadp_clients)}/{len(gov)} -> {tadp_clients}"
    )

    print_banner("GX CORE NATIVE RAW-TRAIN VALIDATOR BASELINE")
    gx_base_cols = [
        "client", "gx_native_suite_success", "ge_expectations_passed",
        "ge_expectations_total", "ge_pass_rate", "ge_native_validation_class",
        "ge_final_action",
    ]
    gx_category_cols = [
        c for c in ge.columns
        if c.startswith("ge_") and (c.endswith("_passed") or c.endswith("_total"))
        and c not in {"ge_expectations_passed", "ge_expectations_total"}
    ]
    print(ge[gx_base_cols + sorted(gx_category_cols)].to_string(index=False))
    print(
        f"\nGX operationally valid clients: "
        f"{len(ge_clients)}/{len(ge)} -> {ge_clients}"
    )

def _write_csv_synced(df: pd.DataFrame, path: Path) -> Dict[str, float]:
    """
    Serialize a CSV, flush Python buffers, request an OS-level fsync, and return
    application-level elapsed time plus final on-disk byte size.

    This is NOT claimed to be raw physical-device latency because Colab storage,
    filesystem caching, and virtualization remain outside the experiment.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    t0 = time.perf_counter()
    with open(path, "w", encoding="utf-8", newline="") as fh:
        df.to_csv(fh, index=False)
        fh.flush()
        os.fsync(fh.fileno())
    elapsed = float(time.perf_counter() - t0)

    return {
        "write_time_s": elapsed,
        "file_bytes": int(path.stat().st_size),
    }


def _hash_chain_rows(event_rows: List[Dict[str, Any]], ledger_scope: str) -> pd.DataFrame:
    """Create a SHA-256 forward hash chain over canonicalized governance events."""
    rows = []
    prev_hash = "GENESIS"

    for seq, event in enumerate(event_rows, start=1):
        payload = dict(event)
        payload["sequence"] = int(seq)
        payload["ledger_scope"] = str(ledger_scope)
        payload["previous_hash"] = str(prev_hash)

        canonical = json.dumps(
            payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False
        )
        entry_hash = hashlib.sha256(canonical.encode("utf-8")).hexdigest()
        payload["entry_hash"] = entry_hash
        rows.append(payload)
        prev_hash = entry_hash

    return pd.DataFrame(rows)


def build_governance_ledgers_with_metrics(
    gov: pd.DataFrame,
    ge: pd.DataFrame,
    root: Path,
) -> Dict[str, Any]:
    """
    Materialize the ledger architecture described in the paper.

    TADP:
      1) one local append-only ledger file per submitted client;
      2) one central/server aggregate ledger containing every TADP decision;
      3) cumulative serialized-byte growth after each central entry.

    Great Expectations:
      separate central technical-validator ledger; it is not mixed into the
      TADP hash chain.

    Governance is executed once before the five training seeds, so A1 records
    one governance event per submitted client. Longitudinal local-ledger growth
    would require repeated governance cycles/versions and is outside A1.
    """
def build_governance_ledgers_with_metrics(
    gov: pd.DataFrame,
    ge: pd.DataFrame,
    root: Path,
) -> Dict[str, Any]:
    """
    Materialize the ledger architecture described in the paper.

    TADP:
      1) one local append-only ledger file per submitted client;
      2) one central/server aggregate ledger containing every TADP decision;
      3) cumulative serialized-byte growth after each central entry.

    Great Expectations:
      separate central technical-validator ledger; it is not mixed into the
      TADP hash chain.

    Governance is executed once before the five training seeds, so A1 records
    one governance event per submitted client. Longitudinal local-ledger growth
    would require repeated governance cycles/versions and is outside A1.
    """
    t_ledger_total0 = time.perf_counter()

    root = Path(root)
    local_dir = root / "ledgers" / "local"
    central_dir = root / "ledgers" / "central"
    local_dir.mkdir(parents=True, exist_ok=True)
    central_dir.mkdir(parents=True, exist_ok=True)

    # -----------------------------
    # TADP event records
    # -----------------------------
    tadp_events = []
    for _, r in gov.sort_values("client").iterrows():
        tadp_events.append({
            "governance_system": "TADP",
            "client": str(r["client"]),
            "hps": float(r["hps"]) if pd.notna(r["hps"]) else None,
            "hps_review": (
                float(r["hps_review"]) if pd.notna(r["hps_review"]) else None
            ),
            "hps_review_midpoint": float(r["hps_review_midpoint"]),
            "evidence_validity_pass": bool(r["evidence_validity_pass"]),
            "evidence_nonpositive": str(r["evidence_nonpositive"]),
            "dimension_floor_failures": str(r["dimension_floor_failures"]),
            "decision_path": str(r["decision_path"]),
            "initial_action": str(r["initial_action"]),
            "final_action": str(r["final_action"]),
            "status": str(r["status"]),
            "reason": str(r["reason"]),
        })

    # Central/server chain across all TADP submissions.
    tadp_central_df = _hash_chain_rows(
        tadp_events, ledger_scope="CENTRAL_SERVER_TADP"
    )
    central_tadp_path = central_dir / "tadp_governance_ledger.csv"
    central_write = _write_csv_synced(tadp_central_df, central_tadp_path)

    # Central cumulative serialized-size growth after each appended governance event.
    growth_rows = []
    prev_bytes = 0
    for n in range(1, len(tadp_central_df) + 1):
        prefix = tadp_central_df.iloc[:n]
        serialized_b = len(prefix.to_csv(index=False).encode("utf-8"))
        growth_rows.append({
            "entry_index": int(n),
            "client": str(tadp_central_df.iloc[n - 1]["client"]),
            "cumulative_serialized_bytes": int(serialized_b),
            "incremental_serialized_bytes": int(serialized_b - prev_bytes),
            "entries": int(n),
        })
        prev_bytes = serialized_b

    growth_df = pd.DataFrame(growth_rows)
    growth_df.to_csv(
        root / "ledger_central_growth.csv",
        index=False,
    )

    # One local ledger per client. In A1 each contains one governance event.
    local_metric_rows = []
    for event in tadp_events:
        cid = str(event["client"])
        local_df = _hash_chain_rows(
            [event], ledger_scope=f"LOCAL_CLIENT_{cid}"
        )
        local_path = local_dir / f"{cid}_ledger.csv"
        local_write = _write_csv_synced(local_df, local_path)
        local_metric_rows.append({
            "client": cid,
            "entries": 1,
            "file_bytes": int(local_write["file_bytes"]),
            "write_time_s": float(local_write["write_time_s"]),
            "bytes_per_entry": int(local_write["file_bytes"]),
            "path": str(local_path),
        })

    local_metrics_df = pd.DataFrame(local_metric_rows).sort_values("client")
    local_metrics_df.to_csv(
        root / "ledger_local_metrics.csv",
        index=False,
    )

    # -----------------------------
    # Great Expectations ledger
    # -----------------------------
    ge_events = []
    for _, r in ge.sort_values("client").iterrows():
        ge_events.append({
            "governance_system": "Great Expectations GX Core native validator",
            "client": str(r["client"]),
            "decision_path": "RULE_BASED_VALIDATION",
            "initial_action": "RULE_BASED_VALIDATION",
            "final_action": str(r["ge_final_action"]),
            "status": "GX_CONTROLLED_RAW_TRAIN_ONLY",
            "reason": (
                f"GX native suite "
                f"{'PASSED' if bool(r['gx_native_suite_success']) else 'FAILED'}; "
                f"{int(r['ge_expectations_passed'])}/"
                f"{int(r['ge_expectations_total'])} configured technical "
                "Expectations passed"
            ),
        })

    ge_df = _hash_chain_rows(
        ge_events, ledger_scope="CENTRAL_SERVER_GX"
    )
    ge_path = central_dir / "gx_core_ledger.csv"
    ge_write = _write_csv_synced(ge_df, ge_path)

    local_total_bytes = int(local_metrics_df["file_bytes"].sum())
    local_total_write_s = float(local_metrics_df["write_time_s"].sum())

    tadp_ledger_total_elapsed_s = float(
        time.perf_counter() - t_ledger_total0
    )

    summary = {
        "tadp_central_entries": int(len(tadp_central_df)),
        "tadp_central_file_bytes": int(central_write["file_bytes"]),
        "tadp_central_write_time_s": float(central_write["write_time_s"]),
        "tadp_local_ledger_count": int(len(local_metrics_df)),
        "tadp_local_total_bytes": local_total_bytes,
        "tadp_local_mean_bytes": float(local_metrics_df["file_bytes"].mean()),
        "tadp_local_min_bytes": int(local_metrics_df["file_bytes"].min()),
        "tadp_local_max_bytes": int(local_metrics_df["file_bytes"].max()),
        "tadp_local_total_write_time_s": local_total_write_s,
        "tadp_local_mean_write_time_s": float(
            local_metrics_df["write_time_s"].mean()
        ),
        "gx_central_entries": int(len(ge_df)),
        "gx_central_file_bytes": int(ge_write["file_bytes"]),
        "gx_central_write_time_s": float(ge_write["write_time_s"]),
        "tadp_ledger_total_elapsed_s": tadp_ledger_total_elapsed_s,
        "write_time_scope": (
            "application-level CSV serialization + flush + fsync request; "
            "not raw physical-storage latency"
        ),
        "size_scope": "final UTF-8 CSV bytes on disk",
    }

    pd.DataFrame([summary]).to_csv(
        root / "ledger_storage_write_summary.csv",
        index=False,
    )
    atomic_write_json(
        summary,
        root / "ledger_storage_write_summary.json",
    )

    return {
        "tadp_central": tadp_central_df,
        "tadp_local_metrics": local_metrics_df,
        "tadp_central_growth": growth_df,
        "gx_central": ge_df,
        "summary": summary,
    }

def choose_experiment_root(

experiment_name, use_drive=True):
    if use_drive:
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
            root = Path("/content/drive/MyDrive/TADP_CHECKPOINTS") / experiment_name
            root.mkdir(parents=True, exist_ok=True)
            print(f"Persistent checkpoint root: {root}")
            return root
        except Exception as exc:
            print(f"Google Drive checkpoint mount unavailable: {exc}")
    root = Path("/content") / experiment_name
    root.mkdir(parents=True, exist_ok=True)
    print(f"Local checkpoint root: {root}")
    return root


def atomic_write_json(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")
    tmp.replace(path)


def load_checkpoint_state(path):
    path = Path(path)
    if not path.exists():
        return {"completed": [], "last_completed": None, "updated_utc": None}
    return json.loads(path.read_text(encoding="utf-8"))


def checkpoint_row_is_compatible(
    perf_path,
    run_idx,
    seed,
    scenario,
    expected_input_dim,
    expected_preprocessor_sha256,
    expected_optimizer_steps,
    expected_fl_rounds,
):
    """
    Reuse a saved scenario only when it matches the current A1 configuration.

    This prevents an old run+scenario checkpoint from being silently reused
    after preprocessing, budget, governance, or resource-accounting changes.
    """
    perf_path = Path(perf_path)
    if not perf_path.exists():
        return False, "performance checkpoint CSV is missing"

    try:
        df = pd.read_csv(perf_path)
    except Exception as exc:
        return False, f"performance checkpoint could not be read: {exc}"

    required = {
        "run", "seed", "scenario",
        "optimizer_steps",
        "input_dim", "preprocessor_sha256",
        "option_A_post_selection_preprocessing",
        "frozen_evidence_assignment_seed",
        "communication_payload_bytes",
        "reference_avg_compute_power_w",
        "reference_grid_co2_kg_per_kwh",
        "reference_electricity_usd_per_kwh",
        "reference_communication_usd_per_mb",
        "fl_rounds",
    }
    missing = sorted(required.difference(df.columns))
    if missing:
        return False, "missing current-result fields: " + ", ".join(missing)

    rows = df[
        df["run"].astype(str).eq(str(run_idx))
        & df["scenario"].astype(str).eq(str(scenario))
    ]
    if len(rows) != 1:
        return False, f"expected exactly one saved row, found {len(rows)}"

    r = rows.iloc[0]

    def _boolish(v):
        if isinstance(v, (bool, np.bool_)):
            return bool(v)
        return str(v).strip().lower() in {"true", "1", "yes"}

    checks = [
        (int(r["seed"]) == int(seed), "training seed changed"),
        (
            int(r["optimizer_steps"]) == int(expected_optimizer_steps),
            "optimizer-step budget changed",
        ),
        (
            int(r["input_dim"]) == int(expected_input_dim),
            "input dimension changed",
        ),
        (
            str(r["preprocessor_sha256"]) == str(expected_preprocessor_sha256),
            "scenario preprocessing changed",
        ),
        (
            _boolish(r["option_A_post_selection_preprocessing"]),
            "result was not produced with Option-A post-selection preprocessing",
        ),
        (
            int(r["frozen_evidence_assignment_seed"])
            == int(FROZEN_EVIDENCE_ASSIGNMENT_SEED),
            "evidence assignment changed",
        ),
        (
            int(r["fl_rounds"]) == int(expected_fl_rounds),
            "FL-round accounting changed",
        ),
        (
            abs(float(r["reference_avg_compute_power_w"])
                - float(REFERENCE_AVG_COMPUTE_POWER_W)) < 1e-12,
            "reference power assumption changed",
        ),
        (
            abs(float(r["reference_grid_co2_kg_per_kwh"])
                - float(REFERENCE_GRID_CO2_KG_PER_KWH)) < 1e-12,
            "grid-carbon reference changed",
        ),
        (
            abs(float(r["reference_electricity_usd_per_kwh"])
                - float(REFERENCE_ELECTRICITY_USD_PER_KWH)) < 1e-12,
            "electricity-price assumption changed",
        ),
        (
            abs(float(r["reference_communication_usd_per_mb"])
                - float(REFERENCE_COMMUNICATION_USD_PER_MB)) < 1e-12,
            "communication-price assumption changed",
        ),
    ]

    for ok, reason in checks:
        if not ok:
            return False, reason

    if "checkpoint_schema_version" in df.columns and pd.notna(
        r.get("checkpoint_schema_version", np.nan)
    ):
        if int(r["checkpoint_schema_version"]) != int(CHECKPOINT_SCHEMA_VERSION):
            return False, "checkpoint schema version changed"

    return True, "compatible with current A1 configuration"


def mark_checkpoint_complete(state_path, key, extra=None):
    state = load_checkpoint_state(state_path)
    completed = list(state.get("completed", []))
    if key not in completed:
        completed.append(key)
    state["completed"] = completed
    state["last_completed"] = key
    state["updated_utc"] = datetime.now(timezone.utc).isoformat()
    if extra:
        state.update(extra)
    atomic_write_json(state, state_path)


def upsert_csv(row, path, key_cols):
    path = Path(path)
    new = pd.DataFrame([row])
    if path.exists():
        old = pd.read_csv(path)
        if not old.empty:
            mask = pd.Series(True, index=old.index)
            for c in key_cols:
                mask &= old[c].astype(str).eq(str(row[c]))
            old = old.loc[~mask].copy()
            new = pd.concat([old, new], ignore_index=True)
    new.to_csv(path, index=False)


def write_scenario_checkpoint(root, run_idx, scenario, result):
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", scenario).strip("_")
    cdir = ensure_dir(root / "scenario_checkpoints" / f"run_{run_idx:02d}")
    payload = {
        "experiment_version": EXPERIMENT_VERSION,
        "checkpoint_schema_version": int(CHECKPOINT_SCHEMA_VERSION),
        "run": int(run_idx),
        "scenario": scenario,
        "completed_utc": datetime.now(timezone.utc).isoformat(),
        "metrics": result["metrics"],
        "runtime_s": float(result["runtime_s"]),
        "optimizer_steps": int(result["total_optimizer_steps"]),
        "communication_payload_bytes": int(result.get("communication_payload_bytes", 0)),
        "communication_mb": float(result.get("communication_mb", 0.0)),
        "communication_mib": float(result.get("communication_mib", 0.0)),
        "ram_peak_mb": float(result.get("ram_peak_mb", 0.0)),
    }
    atomic_write_json(payload, cdir / f"{safe}.json")
    if "round_audit" in result:
        pd.DataFrame(result["round_audit"]).to_csv(
            cdir / f"{safe}_rounds.csv", index=False
        )


def ci95_mean(values):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan
    if len(x) == 1:
        return float(x[0]), float(x[0])
    mean = float(np.mean(x))
    sd = float(np.std(x, ddof=1))
    try:
        from scipy.stats import t
        crit = float(t.ppf(0.975, df=len(x)-1))
    except Exception:
        crit = 1.96
    half = crit * sd / math.sqrt(len(x))
    return mean-half, mean+half


def summarize_runs_with_ci(perf):
    metrics = [
        "accuracy", "precision_macro", "recall_macro", "f1_macro",
        "roc_auc_ovr_macro", "runtime_s", "energy_wh_per_watt",
        "energy_wh", "energy_kwh", "co2_kg",
        "communication_payload_bytes", "communication_mb", "communication_mib",
        "ram_peak_mb", "ram_delta_mb",
        "optimizer_steps", "participants", "energy_cost_usd",
        "communication_cost_usd", "total_estimated_cost_usd",
    ]
    rows = []
    for scenario, d in perf.groupby("scenario", sort=False):
        row = {"scenario": scenario, "n_runs": int(len(d))}
        for metric in metrics:
            if metric not in d.columns:
                continue
            vals = pd.to_numeric(d[metric], errors="coerce")
            vals = vals[np.isfinite(vals)]
            row[f"{metric}_mean"] = float(vals.mean()) if len(vals) else np.nan
            row[f"{metric}_sd"] = float(vals.std(ddof=1)) if len(vals)>1 else 0.0
            lo, hi = ci95_mean(vals)
            row[f"{metric}_ci95_low"] = lo
            row[f"{metric}_ci95_high"] = hi
        rows.append(row)
    return pd.DataFrame(rows)


def paired_tadp_randomk_statistics(perf):
    """Paired five-seed statistics for the strictly budget-matched TADP vs Random-K comparison."""
    tadp = perf[perf["scenario"].eq("TADP Federated")].copy()
    rk = perf[perf["scenario"].eq("Random-K")].copy()
    merged = tadp.merge(
        rk,
        on=["run", "seed"],
        suffixes=("_tadp", "_randomk"),
        validate="one_to_one",
    )
    rows = []
    for metric in ["accuracy", "f1_macro", "roc_auc_ovr_macro"]:
        a = merged[f"{metric}_tadp"].to_numpy(dtype=float)
        b = merged[f"{metric}_randomk"].to_numpy(dtype=float)
        diff = a - b
        mean = float(np.mean(diff))
        sd = float(np.std(diff, ddof=1)) if len(diff) > 1 else 0.0
        lo, hi = ci95_mean(diff)
        t_stat = t_p = wil_stat = wil_p = np.nan
        try:
            from scipy.stats import ttest_rel, wilcoxon
            tr = ttest_rel(a, b, nan_policy="omit")
            t_stat, t_p = float(tr.statistic), float(tr.pvalue)
            if np.any(np.abs(diff) > 0):
                wr = wilcoxon(a, b)
                wil_stat, wil_p = float(wr.statistic), float(wr.pvalue)
        except Exception:
            pass
        rows.append({
            "metric": metric,
            "n_pairs": len(diff),
            "mean_difference_tadp_minus_randomk": mean,
            "sd_difference": sd,
            "ci95_low": lo,
            "ci95_high": hi,
            "cohens_dz": float(mean / sd) if sd > 0 else np.nan,
            "paired_t_stat": t_stat,
            "paired_t_p": t_p,
            "wilcoxon_stat": wil_stat,
            "wilcoxon_p": wil_p,
            "tadp_wins": int(np.sum(diff > 0)),
            "ties": int(np.sum(np.isclose(diff, 0))),
            "tadp_losses": int(np.sum(diff < 0)),
        })
    return pd.DataFrame(rows)


def scenario_method_table():
    return pd.DataFrame([
        ["Naïve Centralized", "centralized", "governance-off baseline",
         "all clients; exact optimizer-step parity with Vanilla FedAvg", "5-seed headline"],
        ["TADP Centralized", "centralized", "TADP",
         "TADP-admitted cohort; exact total optimizer-step parity with TADP Federated", "5-seed headline"],
        ["Vanilla FedAvg", "federated", "FedAvg",
         "all clients", "5-seed headline"],
        ["FedProx", "federated", "FedProx",
         "all clients", "5-seed headline"],
        ["Random-K", "federated", "matched random control",
         "same K, rounds, and realized optimizer-step budget as TADP", "5-seed headline"],
        ["DQ-only Federated", "federated", "DQ-only",
         "top-K by machine-measured DQ only; same K, rounds, and realized optimizer-step budget as TADP", "5-seed headline"],
        ["Power-of-Choice", "federated", "Power-of-Choice",
         "dynamic TRAIN-loss selection; same K, rounds, and realized optimizer-step budget as TADP", "5-seed headline"],
        ["TADP Federated", "federated", "TADP",
         "TADP-admitted cohort", "5-seed headline"],
        ["GX Core", "governance-only", "native technical validator",
         "raw TRAIN shards", "validation audit only"],
    ], columns=["scenario", "paradigm", "method", "participation", "execution_role"])

def governance_only_monte_carlo(


    client_ids,
    dq_scores,
    base_seed,
    n_realizations=1000,
    domain="healthcare",
):
    domain = str(domain).lower()
    rows = []
    for j in range(int(n_realizations)):
        seed = int(base_seed + j)
        evidence, _ = generate_controlled_documentary_evidence(
            client_ids,
            seed,
            domain=domain,
        )
        gov = build_tadp_governance(
            client_ids,
            evidence,
            dq_scores,
            run=0,
            evidence_seed=seed,
            domain=domain,
        )
        accepted = accepted_tadp(gov)
        rows.append({
            "realization": j + 1,
            "evidence_seed": seed,
            "accepted_count": len(accepted),
            "accepted_clients": ";".join(accepted),
            "mean_hps": float(gov["hps"].mean(skipna=True)),
            "mean_hps_review": float(
                gov["hps_review"].mean(skipna=True)
            ),
            "evidence_validity_rejects": int(
                gov["status"].eq(
                    "AUTO_REJECTED_EVIDENCE_VALIDITY"
                ).sum()
            ),
            "dimension_floor_rejects": int(
                gov["status"].eq(
                    "AUTO_REJECTED_DIMENSION_FLOOR"
                ).sum()
            ),
            "low_hps_rejects": int(
                gov["status"].eq(
                    "AUTO_REJECTED_LOW_HPS"
                ).sum()
            ),
            "review_accepts": int(
                gov["status"].eq(
                    "ACCEPTED_AFTER_AUTOMATED_REVIEW"
                ).sum()
            ),
            "review_rejects": int(
                gov["status"].eq(
                    "AUTO_REJECTED_HPS_REVIEW"
                ).sum()
            ),
            "direct_auto_accepts": int(
                gov["status"].eq(
                    "DIRECT_AUTO_ACCEPTED"
                ).sum()
            ),
        })
    return pd.DataFrame(rows)

# ======================================================================================
# TADP EXPERIMENT A

# ======================================================================================
# ======================================================================================
EXPERIMENT_VERSION = "TADP-A1-FINAL-28F-DIMFLOOR-4D-HPSREVIEW-OPTIONA-STEPPARITY-K10-5SEED"
CHECKPOINT_SCHEMA_VERSION = 2
TRAINING_RUN_SEEDS = [42, 142, 242, 342, 442]
NUM_CLIENTS = 10
NUM_ROUNDS_FL = 4
LOCAL_EPOCHS = 1
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
FEDPROX_MU = 0.01
DIRICHLET_ALPHA = 1.0
GLOBAL_SPLIT_SEED = 7001
CLIENT_PARTITION_SEED = 7101
FROZEN_EVIDENCE_ASSIGNMENT_SEED = 1042
FROZEN_RANDOMK_SELECTION_SEED = 2042
USE_GOOGLE_DRIVE_CHECKPOINTS = True
RUN_GOVERNANCE_MONTE_CARLO = True
# Set True to stop after frozen TADP governance/preflight outputs and ZIP them.
# Useful for checking the new admission policy before the long five-seed training run.
GOVERNANCE_PREFLIGHT_ONLY = False

EXPERIMENT_ROOT = choose_experiment_root(
    "TADP_EXPERIMENT_A1_" + EXPERIMENT_VERSION,
    use_drive=USE_GOOGLE_DRIVE_CHECKPOINTS,
)
CHECKPOINT_STATE = EXPERIMENT_ROOT / "checkpoint_state.json"
PERF_CHECKPOINT = EXPERIMENT_ROOT / "performance_metrics_checkpoint.csv"


def main():
    csv_path = locate_diabetes_csv()

    print_banner(EXPERIMENT_VERSION)
    print(f"Training seeds: {TRAINING_RUN_SEEDS}")
    print(f"K={NUM_CLIENTS} raw TRAIN clients | FL rounds={NUM_ROUNDS_FL}")
    print(
        "FINAL TADP policy: 28-factor validity gate -> 6-dimension floor -> "
        "6-dimension HPS -> 4-critical-dimension HPS_review for borderline cases."
    )
    print(
        f"Evidence validity: all 28 scores must be present, finite, and > "
        f"{EVIDENCE_VALIDITY_MIN_EXCLUSIVE:.1f}."
    )
    print(
        f"Dimension floor: every dimension >= {DIMENSION_MIN_FLOOR:.1f}/5."
    )
    print(
        f"Main HPS: <{GOOD_CUT:.1f}=Reject; "
        f"[{GOOD_CUT:.1f},{HIGH_CUT:.1f})=Review; "
        f">={HIGH_CUT:.1f}=Direct Accept."
    )
    print(
        "Review dimensions: "
        + ", ".join(
            f"{d}:{DIMENSION_NAMES[d]}" for d in CRITICAL_REVIEW_DIMS
        )
    )
    print(
        "Review weights (renormalized original HPS weights): "
        + json.dumps(CRITICAL_REVIEW_WEIGHTS, sort_keys=True)
    )
    print(
        f"HPS_review >= midpoint {REVIEW_MIDPOINT:.2f} -> Accept after Review; "
        "otherwise Reject."
    )
    print(
        "OPTION A preprocessing: governance first; model scaler, categorical "
        "vocabulary, and class weights are fitted only after the participant "
        "cohort for a scenario is frozen."
    )

    # ------------------------------------------------------------------
    # STAGE 1 — GLOBAL HOLDOUT + RAW TRAIN SHARDS + DQ GOVERNANCE INPUTS
    # ------------------------------------------------------------------
    data = prepare_diabetes_governance_stage(
        csv_path=csv_path,
        split_seed=GLOBAL_SPLIT_SEED,
        partition_seed=CLIENT_PARTITION_SEED,
        n_clients=NUM_CLIENTS,
        alpha=DIRICHLET_ALPHA,
    )

    leakage = write_leakage_audit(
        EXPERIMENT_ROOT,
        data["global_train_ids"],
        data["global_test_ids"],
        data["client_train_ids"],
        extra={
            "patient_overlap": data["meta"]["patient_overlap"],
            "global_holdout_before_client_partition": True,
            "dq_tadp_use_raw_train_only": True,
            "model_preprocessing_before_governance": False,
            "class_weights_before_governance": False,
            "test_used_for_final_evaluation_only": True,
            "option_A_post_admission_preprocessing": True,
        },
    )
    print_banner("NO-LEAKAGE / ORDERING AUDIT")
    print(pd.DataFrame([leakage]).to_string(index=False))

    pd.DataFrame([data["meta"]]).to_csv(
        EXPERIMENT_ROOT / "dataset_metadata.csv", index=False
    )
    partition_df = client_partition_table(data["clients_raw"])
    partition_df.to_csv(
        EXPERIMENT_ROOT / "client_partition_audit.csv", index=False
    )
    data["dq_audit"].to_csv(
        EXPERIMENT_ROOT / "dq_raw_train_governance_audit.csv",
        index=False,
    )

    print_banner("FROZEN RAW TRAIN CLIENT PARTITIONS")
    print(partition_df.to_string(index=False))

    client_ids = list(map(str, data["client_ids"]))
    all_clients = list(client_ids)

    # Zero-width X arrays are used ONLY for exact optimizer-step budgeting.
    # They contain no model features and consume negligible memory.
    step_proxy_arrays = {
        cid: (
            np.empty((len(data["clients_raw"][cid]), 0), dtype=np.float32),
            data["clients_raw"][cid]["_target"].to_numpy(dtype=np.int32),
        )
        for cid in client_ids
    }

    # ------------------------------------------------------------------
    # STAGE 2 — DOCUMENTARY EVIDENCE + FINAL TADP GOVERNANCE
    # ------------------------------------------------------------------
    t_evidence0 = time.perf_counter()
    documentary_evidence, evidence_df = (
        generate_controlled_documentary_evidence(
            client_ids,
            FROZEN_EVIDENCE_ASSIGNMENT_SEED,
            domain="healthcare",
        )
    )
    evidence_df.to_csv(
        EXPERIMENT_ROOT / "controlled_documentary_evidence_matrix.csv",
        index=False,
    )
    evidence_summary = evidence_assignment_summary(evidence_df)
    evidence_summary.to_csv(
        EXPERIMENT_ROOT / "evidence_assignment_summary.csv",
        index=False,
    )
    evidence_processing_s = float(time.perf_counter() - t_evidence0)

    t_policy0 = time.perf_counter()
    gov = build_tadp_governance(
        client_ids,
        documentary_evidence,
        data["dq_scores"],
        run=0,
        evidence_seed=FROZEN_EVIDENCE_ASSIGNMENT_SEED,
        domain="healthcare",
    )
    policy_evaluation_s = float(time.perf_counter() - t_policy0)
    gov.to_csv(
        EXPERIMENT_ROOT / "tadp_governance_frozen.csv",
        index=False,
    )

    full_evidence = build_full_factor_evidence_table(
        client_ids=client_ids,
        documentary_evidence=documentary_evidence,
        dq_scores=data["dq_scores"],
        dq_audit=data["dq_audit"],
        evidence_seed=FROZEN_EVIDENCE_ASSIGNMENT_SEED,
        domain="healthcare",
    )
    full_evidence.to_csv(
        EXPERIMENT_ROOT / "all_28_factor_evidence_by_client.csv",
        index=False,
    )

    policy_rows = []
    for dim in FACTOR_NAMES:
        for factor in FACTOR_NAMES[dim]:
            policy_rows.append({
                "domain": "healthcare",
                "dimension": dim,
                "dimension_name": DIMENSION_NAMES[dim],
                "factor": factor,
                "factor_count_in_dimension": len(FACTOR_NAMES[dim]),
                "evidence_validity_rule": "present_finite_and_strictly_gt_0",
                "dimension_floor_0_5": float(DIMENSION_MIN_FLOOR),
                "main_hps_weight": float(
                    WEIGHTS_PSCORE_DEFAULT[dim]
                ),
                "is_critical_review_dimension": bool(
                    dim in CRITICAL_REVIEW_DIMS
                ),
                "review_hps_weight": (
                    float(CRITICAL_REVIEW_WEIGHTS[dim])
                    if dim in CRITICAL_REVIEW_DIMS
                    else np.nan
                ),
                "main_hps_reject_below": float(GOOD_CUT),
                "main_hps_direct_accept_at_or_above": float(HIGH_CUT),
                "hps_review_midpoint": float(REVIEW_MIDPOINT),
                "hps_review_accept_rule": "greater_than_or_equal_to_midpoint",
            })
    pd.DataFrame(policy_rows).to_csv(
        EXPERIMENT_ROOT / "healthcare_admission_policy.csv",
        index=False,
    )

    tadp_clients = accepted_tadp(gov)
    rejected_tadp_clients = sorted(
        gov.loc[
            gov["final_action"].eq("REJECT"), "client"
        ].astype(str).tolist()
    )
    if not tadp_clients:
        raise RuntimeError("TADP admitted zero clients. Fail closed.")

    # Branch preflight.
    branch_audit = (
        evidence_summary[
            [
                "client", "bundle_id", "evidence_profile",
                "scenario_role", "profile_variant",
            ]
        ]
        .merge(
            gov[
                [
                    "client", "evidence_validity_pass", "hps",
                    "hps_review", "hps_review_midpoint",
                    "dimension_floor_failures", "decision_path",
                    "final_action", "status", "reason",
                ]
            ],
            on="client",
            how="left",
            validate="one_to_one",
        )
        .sort_values("client")
        .reset_index(drop=True)
    )
    branch_audit.to_csv(
        EXPERIMENT_ROOT / "governance_branch_preflight_audit.csv",
        index=False,
    )
    print_banner("FINAL GOVERNANCE PREFLIGHT — BEFORE MODEL PREPROCESSING")
    print(branch_audit.to_string(index=False))
    print(
        f"TADP admitted {len(tadp_clients)}/{len(client_ids)}: "
        f"{tadp_clients}"
    )
    print(f"TADP rejected: {rejected_tadp_clients}")

    expected_role_outcomes = {
        "DIRECT_STRONG": ("DIRECT_AUTO_ACCEPT", "ACCEPT"),
        "REVIEW_RECOVERABLE": ("HPS_REVIEW_BAND", "ACCEPT"),
        "REVIEW_LIMITED": ("HPS_REVIEW_BAND", "REJECT"),
        "EVIDENCE_INVALID": ("EVIDENCE_VALIDITY_GATE", "REJECT"),
        "LOW_HPS_WEAK": ("LOW_HPS", "REJECT"),
        "DIMENSION_FLOOR_WEAK": ("DIMENSION_FLOOR", "REJECT"),
    }
    role_check_rows = []
    for role, (expected_path, expected_action) in (
        expected_role_outcomes.items()
    ):
        d = branch_audit.loc[
            branch_audit["scenario_role"].eq(role)
        ]
        if d.empty:
            continue
        pass_ = bool(
            d["decision_path"].eq(expected_path).all()
            and d["final_action"].eq(expected_action).all()
        )
        role_check_rows.append({
            "scenario_role": role,
            "expected_decision_path": expected_path,
            "expected_final_action": expected_action,
            "observed_paths": ";".join(
                sorted(d["decision_path"].astype(str).unique())
            ),
            "observed_actions": ";".join(
                sorted(d["final_action"].astype(str).unique())
            ),
            "pass": pass_,
        })
    role_check_df = pd.DataFrame(role_check_rows)
    role_check_df.to_csv(
        EXPERIMENT_ROOT / "governance_branch_role_check.csv",
        index=False,
    )
    if not role_check_df.empty and not role_check_df["pass"].all():
        raise RuntimeError(
            "Controlled governance branch preflight failed. "
            "Inspect governance_branch_role_check.csv."
        )

    if GOVERNANCE_PREFLIGHT_ONLY:
        preflight_policy = {
            "experiment_version": EXPERIMENT_VERSION,
            "preflight_only": True,
            "total_factors": 28,
            "evidence_validity_rule": "all_28_present_finite_and_gt_0",
            "dimension_min_floor": DIMENSION_MIN_FLOOR,
            "HPS_reject_threshold": GOOD_CUT,
            "HPS_accept_threshold": HIGH_CUT,
            "critical_review_dimensions": list(CRITICAL_REVIEW_DIMS),
            "critical_review_weights": CRITICAL_REVIEW_WEIGHTS,
            "HPS_review_midpoint": REVIEW_MIDPOINT,
            "HPS_review_accept_rule": "greater_than_or_equal_to_midpoint",
            "tadp_clients": tadp_clients,
            "tadp_rejected_clients": rejected_tadp_clients,
            "option_A_post_admission_preprocessing": True,
        }
        atomic_write_json(
            preflight_policy,
            EXPERIMENT_ROOT / "governance_preflight_policy.json",
        )
        return EXPERIMENT_ROOT

    # ------------------------------------------------------------------
    # GX RAW-TRAIN VALIDATOR — separate comparator, still before training
    # ------------------------------------------------------------------
    ge, ge_expectation_audit = ge_governance(
        client_data=data["clients_raw"],
        client_ids=client_ids,
        accept_count=len(tadp_clients),
        domain="healthcare",
        preprocessor=data["dq_schema"],
        dq_scores=data["dq_scores"],
    )
    ge.to_csv(
        EXPERIMENT_ROOT / "gx_core_raw_train_governance_frozen.csv",
        index=False,
    )
    ge_expectation_audit.to_csv(
        EXPERIMENT_ROOT / "gx_core_expectation_audit.csv",
        index=False,
    )
    ge_clients = ge.loc[
        ge["ge_final_action"].eq("ACCEPT"), "client"
    ].astype(str).tolist()

    # ------------------------------------------------------------------
    # MATCHED CONTROL COHORTS
    # ------------------------------------------------------------------
    rk_rng = np.random.default_rng(FROZEN_RANDOMK_SELECTION_SEED)
    randomk_clients = rk_rng.choice(
        client_ids, size=len(tadp_clients), replace=False
    ).tolist()

    dq_only_clients, dq_only_audit = select_dq_only_clients(
        data["dq_scores"], len(tadp_clients)
    )
    dq_only_audit.to_csv(
        EXPERIMENT_ROOT / "dq_only_selection_audit.csv",
        index=False,
    )

    if len(randomk_clients) != len(tadp_clients):
        raise RuntimeError("Random-K K does not match TADP K.")

    # Main participation sets. For Power-of-Choice, all clients form
    # the candidate universe, therefore its preprocessing is fitted on that
    # candidate universe before dynamic per-round loss selection.
    cohorts = {
        "Naïve Centralized": all_clients,
        "TADP Centralized": tadp_clients,
        "Vanilla FedAvg": all_clients,
        "FedProx": all_clients,
        "Random-K": randomk_clients,
        "DQ-only Federated": dq_only_clients,
        "Power-of-Choice": all_clients,
        "TADP Federated": tadp_clients,
    }

    # ------------------------------------------------------------------
    # STAGE 3 — OPTION A: FIT MODEL PREPROCESSING AFTER PARTICIPATION
    # ------------------------------------------------------------------
    scenario_views, preprocessing_audit = build_training_view_cache(
        clients_raw=data["clients_raw"],
        test_df=data["test_df"],
        cohorts=cohorts,
    )
    preprocessing_audit.to_csv(
        EXPERIMENT_ROOT / "post_selection_preprocessing_audit.csv",
        index=False,
    )

    # Strong TADP-specific fail-closed audit.
    for scenario in ["TADP Centralized", "TADP Federated"]:
        view_clients = set(
            scenario_views[scenario]["participating_clients"]
        )
        if view_clients != set(tadp_clients):
            raise RuntimeError(
                f"Option A violation in {scenario}: preprocessing cohort "
                "does not exactly equal TADP accepted cohort."
            )
        if view_clients & set(rejected_tadp_clients):
            raise RuntimeError(
                f"Option A violation in {scenario}: rejected TADP clients "
                "entered preprocessing."
            )

    option_a_audit = {
        "policy": "OPTION_A_GOVERNANCE_BEFORE_MODEL_PREPROCESSING",
        "raw_train_governance_first": True,
        "tadp_accepted_clients": sorted(tadp_clients),
        "tadp_rejected_clients": rejected_tadp_clients,
        "tadp_preprocessing_fit_clients": sorted(
            scenario_views["TADP Federated"][
                "participating_clients"
            ]
        ),
        "rejected_clients_in_tadp_preprocessing": sorted(
            set(rejected_tadp_clients)
            & set(
                scenario_views["TADP Federated"][
                    "participating_clients"
                ]
            )
        ),
        "tadp_preprocessor_sha256": (
            scenario_views["TADP Federated"][
                "preprocessor_sha256"
            ]
        ),
        "tadp_class_weights": (
            scenario_views["TADP Federated"]["class_weights"]
        ),
        "pass": True,
    }
    atomic_write_json(
        option_a_audit,
        EXPERIMENT_ROOT / "option_A_preprocessing_order_audit.json",
    )

    frozen_plan = {
        "experiment_version": EXPERIMENT_VERSION,
        "training_seeds": TRAINING_RUN_SEEDS,
        "fl_rounds": NUM_ROUNDS_FL,
        "global_split_seed": GLOBAL_SPLIT_SEED,
        "client_partition_seed": CLIENT_PARTITION_SEED,
        "frozen_evidence_assignment_seed": FROZEN_EVIDENCE_ASSIGNMENT_SEED,
        "frozen_randomk_selection_seed": FROZEN_RANDOMK_SELECTION_SEED,
        "all_clients": all_clients,
        "ge_clients": ge_clients,
        "tadp_clients": tadp_clients,
        "tadp_rejected_clients": rejected_tadp_clients,
        "randomk_clients": randomk_clients,
        "dq_only_clients": dq_only_clients,
        "evidence_validity_rule": "all_28_present_finite_and_gt_0",
        "critical_review_dimensions": list(CRITICAL_REVIEW_DIMS),
        "critical_review_weights": CRITICAL_REVIEW_WEIGHTS,
        "HPS_review_midpoint": float(REVIEW_MIDPOINT),
        "HPS_review_accept_rule": "greater_than_or_equal_to_midpoint",
        "option_A_post_selection_preprocessing": True,
        "gx_selection_semantics": (
            "GX severity-aware validator; zero critical failures required; "
            "no ranking; no forced-K"
        ),
        "power_of_choice_semantics": (
            "dynamic TRAIN-loss selection per round from all-client "
            "candidate universe"
        ),
        "dq_only_semantics": (
            "top-K by mean of the eight machine-measured DQ factors; "
            "K, FL rounds, and realized optimizer-step budget matched to TADP"
        ),
    }
    plan_path = EXPERIMENT_ROOT / "frozen_cohort_plan.json"
    atomic_write_json(frozen_plan, plan_path)

    print_governance_details(
        gov,
        ge,
        data["dq_audit"],
        tadp_clients,
        ge_clients,
        domain="healthcare",
    )

    ledger_artifacts = build_governance_ledgers_with_metrics(
        gov,
        ge,
        EXPERIMENT_ROOT,
    )
    print_banner("LEDGER STORAGE / WRITE AUDIT")
    print(
        pd.DataFrame([ledger_artifacts["summary"]]).to_string(index=False)
    )

    governance_timing = {
        "raw_load_split_partition_s": float(
            data["meta"]["raw_load_split_partition_s"]
        ),
        "dq_measurement_s": float(data["meta"]["dq_measurement_s"]),
        "documentary_evidence_software_processing_s": float(
            evidence_processing_s
        ),
        "policy_evaluation_s": float(policy_evaluation_s),
        "tadp_ledger_total_elapsed_s": float(
            ledger_artifacts["summary"]["tadp_ledger_total_elapsed_s"]
        ),
    }
    governance_timing["tadp_governance_software_total_s"] = float(
        governance_timing["dq_measurement_s"]
        + governance_timing["documentary_evidence_software_processing_s"]
        + governance_timing["policy_evaluation_s"]
        + governance_timing["tadp_ledger_total_elapsed_s"]
    )
    governance_timing["human_evidence_verification_included"] = False

    pd.DataFrame([governance_timing]).to_csv(
        EXPERIMENT_ROOT / "governance_software_timing.csv",
        index=False,
    )
    atomic_write_json(
        governance_timing,
        EXPERIMENT_ROOT / "governance_software_timing.json",
    )

    print_banner("GOVERNANCE SOFTWARE TIMING")
    print(pd.DataFrame([governance_timing]).to_string(index=False))
    scenario_method_table().to_csv(
        EXPERIMENT_ROOT / "scenario_definitions.csv",
        index=False,
    )

    print_banner("OPTION A — POST-SELECTION PREPROCESSING AUDIT")
    print(preprocessing_audit.to_string(index=False))
    print(
        "TADP rejected clients are excluded from its scaler, categorical "
        "vocabulary, class weights, and transformed TRAIN arrays."
    )

    # ------------------------------------------------------------------
    # EXACT OPTIMIZER-STEP PLANS — based on realized raw shard sizes
    # ------------------------------------------------------------------
    def natural_map(cohort):
        return {
            c: natural_steps(
                len(data["clients_raw"][c]),
                BATCH_SIZE,
                LOCAL_EPOCHS,
            )
            for c in cohort
        }

    step_map_all = natural_map(all_clients)
    step_map_tadp = natural_map(tadp_clients)
    tadp_target_per_round = int(sum(step_map_tadp.values()))

    step_map_randomk = allocate_exact_step_budget(
        randomk_clients,
        step_proxy_arrays,
        tadp_target_per_round,
        BATCH_SIZE,
        LOCAL_EPOCHS,
    )
    step_map_dq_only = allocate_exact_step_budget(
        dq_only_clients,
        step_proxy_arrays,
        tadp_target_per_round,
        BATCH_SIZE,
        LOCAL_EPOCHS,
    )

    fed_step_maps = {
        "Vanilla FedAvg": [
            dict(step_map_all) for _ in range(NUM_ROUNDS_FL)
        ],
        "FedProx": [
            dict(step_map_all) for _ in range(NUM_ROUNDS_FL)
        ],
        "Random-K": [
            dict(step_map_randomk) for _ in range(NUM_ROUNDS_FL)
        ],
        "DQ-only Federated": [
            dict(step_map_dq_only) for _ in range(NUM_ROUNDS_FL)
        ],
        "TADP Federated": [
            dict(step_map_tadp) for _ in range(NUM_ROUNDS_FL)
        ],
    }

    central_pair = {
        "Naïve Centralized": "Vanilla FedAvg",
        "TADP Centralized": "TADP Federated",
    }

    central_fed_plan_rows = []
    for central_name, fed_name in central_pair.items():
        fed_total = int(
            sum(sum(m.values()) for m in fed_step_maps[fed_name])
        )
        cview = scenario_views[central_name]
        fview = scenario_views[fed_name]
        same_pre = (
            cview["preprocessor_sha256"]
            == fview["preprocessor_sha256"]
        )
        same_dim = int(cview["input_dim"]) == int(fview["input_dim"])
        central_fed_plan_rows.append({
            "centralized_scenario": central_name,
            "federated_pair": fed_name,
            "selected_clients": ";".join(cohorts[central_name]),
            "selected_k": len(cohorts[central_name]),
            "federated_total_optimizer_steps": fed_total,
            "centralized_exact_optimizer_steps": fed_total,
            "exact_step_parity": True,
            "same_post_selection_preprocessor": bool(same_pre),
            "same_input_dim": bool(same_dim),
            "preprocessor_sha256": cview["preprocessor_sha256"],
        })

    central_fed_plan_df = pd.DataFrame(central_fed_plan_rows)
    central_fed_plan_df.to_csv(
        EXPERIMENT_ROOT
        / "centralized_federated_exact_step_and_preprocessing_parity_plan.csv",
        index=False,
    )
    if not central_fed_plan_df[
        [
            "exact_step_parity",
            "same_post_selection_preprocessor",
            "same_input_dim",
        ]
    ].all().all():
        raise RuntimeError(
            "Centralized/federated parity plan failed."
        )

    randomk_parity_rows = []
    for r in range(1, NUM_ROUNDS_FL + 1):
        randomk_parity_rows.append({
            "round": r,
            "tadp_clients": ";".join(tadp_clients),
            "randomk_clients": ";".join(randomk_clients),
            "tadp_k": len(tadp_clients),
            "randomk_k": len(randomk_clients),
            "tadp_steps": tadp_target_per_round,
            "randomk_steps": int(sum(step_map_randomk.values())),
            "same_k": len(tadp_clients) == len(randomk_clients),
            "exact_step_parity": (
                tadp_target_per_round
                == int(sum(step_map_randomk.values()))
            ),
            "tadp_preprocessor_sha256": (
                scenario_views["TADP Federated"][
                    "preprocessor_sha256"
                ]
            ),
            "randomk_preprocessor_sha256": (
                scenario_views["Random-K"]["preprocessor_sha256"]
            ),
            "preprocessing_policy": (
                "each selected cohort fits its own post-selection transform"
            ),
        })
    randomk_parity_df = pd.DataFrame(randomk_parity_rows)
    randomk_parity_df.to_csv(
        EXPERIMENT_ROOT / "tadp_randomk_parity_plan.csv",
        index=False,
    )
    if not randomk_parity_df[
        ["same_k", "exact_step_parity"]
    ].all().all():
        raise RuntimeError("TADP / Random-K budget parity failed.")

    # Strict apples-to-apples selection controls.
    # TADP determines the realized admitted K. Random-K, DQ-only, and
    # Power-of-Choice are matched to that K, the same number of FL rounds,
    # and the same realized optimizer-step budget. The selection signal is
    # therefore the principal controlled difference.
    matched_control_plan = pd.DataFrame([
        {
            "method": "TADP",
            "selected_k_per_round": len(tadp_clients),
            "rounds": NUM_ROUNDS_FL,
            "steps_per_round": tadp_target_per_round,
            "total_optimizer_steps": tadp_target_per_round * NUM_ROUNDS_FL,
            "selection_signal": "28-factor TADP governance",
        },
        {
            "method": "Random-K",
            "selected_k_per_round": len(randomk_clients),
            "rounds": NUM_ROUNDS_FL,
            "steps_per_round": int(sum(step_map_randomk.values())),
            "total_optimizer_steps": int(sum(step_map_randomk.values())) * NUM_ROUNDS_FL,
            "selection_signal": "random selection",
        },
        {
            "method": "DQ-only",
            "selected_k_per_round": len(dq_only_clients),
            "rounds": NUM_ROUNDS_FL,
            "steps_per_round": int(sum(step_map_dq_only.values())),
            "total_optimizer_steps": int(sum(step_map_dq_only.values())) * NUM_ROUNDS_FL,
            "selection_signal": "top-K by mean of 8 machine-measured DQ factors",
        },
        {
            "method": "Power-of-Choice",
            "selected_k_per_round": len(tadp_clients),
            "rounds": NUM_ROUNDS_FL,
            "steps_per_round": tadp_target_per_round,
            "total_optimizer_steps": tadp_target_per_round * NUM_ROUNDS_FL,
            "selection_signal": "dynamic TRAIN-loss utility",
        },
    ])

    matched_control_plan["same_k_as_tadp"] = (
        matched_control_plan["selected_k_per_round"].eq(len(tadp_clients))
    )
    matched_control_plan["same_rounds_as_tadp"] = (
        matched_control_plan["rounds"].eq(NUM_ROUNDS_FL)
    )
    matched_control_plan["same_steps_as_tadp"] = (
        matched_control_plan["total_optimizer_steps"].eq(
            tadp_target_per_round * NUM_ROUNDS_FL
        )
    )

    matched_control_plan.to_csv(
        EXPERIMENT_ROOT / "tadp_selection_control_step_parity_plan.csv",
        index=False,
    )

    if not matched_control_plan[
        ["same_k_as_tadp", "same_rounds_as_tadp", "same_steps_as_tadp"]
    ].all().all():
        raise RuntimeError(
            "Selection-control parity failed for TADP, Random-K, DQ-only, "
            "or Power-of-Choice."
        )

    if RUN_GOVERNANCE_MONTE_CARLO:
        mc = governance_only_monte_carlo(
            client_ids,
            data["dq_scores"],
            base_seed=500000,
            n_realizations=1000,
            domain="healthcare",
        )
        mc.to_csv(
            EXPERIMENT_ROOT
            / "governance_only_1000_evidence_realizations.csv",
            index=False,
        )

    # ------------------------------------------------------------------
    # TRAIN 8 HEADLINE SCENARIOS x 5 SEEDS
    # ------------------------------------------------------------------
    completed = set(
        load_checkpoint_state(CHECKPOINT_STATE).get("completed", [])
    )
    selection_rows = []
    central_fed_runtime_rows = []
    monitor_cache = {}

    def monitor_for_view(view):
        key = view["preprocessor_sha256"]
        if key not in monitor_cache:
            monitor_cache[key] = build_train_monitor_subset(
                view["client_arrays"],
                max_samples=ROUND_PROGRESS_MONITOR_MAX_SAMPLES,
                seed=99117,
            )
        return monitor_cache[key]

    for run_idx, seed in enumerate(
        TRAINING_RUN_SEEDS, start=1
    ):
        print_banner(
            f"RUN {run_idx}/{len(TRAINING_RUN_SEEDS)} | seed={seed}"
        )

        for scenario_idx, scenario in enumerate(
            MANUSCRIPT_SCENARIOS, start=1
        ):
            key = f"run={run_idx}|scenario={scenario}"

            view = scenario_views[scenario]
            selected = list(cohorts[scenario])
            input_dim = int(view["input_dim"])

            if scenario in central_pair:
                paired_fed_for_checkpoint = central_pair[scenario]
                expected_optimizer_steps = int(
                    sum(
                        sum(m.values())
                        for m in fed_step_maps[paired_fed_for_checkpoint]
                    )
                )
                expected_fl_rounds = 0
            elif scenario == "Power-of-Choice":
                expected_optimizer_steps = int(
                    tadp_target_per_round * NUM_ROUNDS_FL
                )
                expected_fl_rounds = int(NUM_ROUNDS_FL)
            else:
                expected_optimizer_steps = int(
                    sum(sum(m.values()) for m in fed_step_maps[scenario])
                )
                expected_fl_rounds = int(NUM_ROUNDS_FL)

            if key in completed:
                compatible, checkpoint_reason = checkpoint_row_is_compatible(
                    PERF_CHECKPOINT,
                    run_idx=run_idx,
                    seed=seed,
                    scenario=scenario,
                    expected_input_dim=input_dim,
                    expected_preprocessor_sha256=view["preprocessor_sha256"],
                    expected_optimizer_steps=expected_optimizer_steps,
                    expected_fl_rounds=expected_fl_rounds,
                )
                if compatible:
                    print(
                        f"CHECKPOINT VALID -> SKIP: {key} | "
                        f"{checkpoint_reason}"
                    )
                    continue

                print(
                    f"STALE CHECKPOINT -> RERUN: {key} | "
                    f"{checkpoint_reason}"
                )
                completed.discard(key)

            build_model_fn = (
                lambda input_dim=input_dim:
                build_diabetes_model(input_dim, LEARNING_RATE)
            )

            # Same run seed + same input dimension => exactly same W0.
            seed_everything(seed)
            init_model = build_model_fn()
            initial_weights = [
                np.array(w, copy=True)
                for w in init_model.get_weights()
            ]
            initial_hash = sha256_weights(initial_weights)
            del init_model
            tf.keras.backend.clear_session()
            gc.collect()

            upsert_csv(
                {
                    "run": run_idx,
                    "seed": seed,
                    "scenario": scenario,
                    "initialization_seed": seed,
                    "input_dim": input_dim,
                    "initial_weights_sha256": initial_hash,
                    "preprocessor_sha256": view[
                        "preprocessor_sha256"
                    ],
                    "preprocessing_fit_clients": ";".join(
                        view["participating_clients"]
                    ),
                },
                EXPERIMENT_ROOT / "initialization_audit.csv",
                ["run", "scenario"],
            )

            progress_context = {
                "run_idx": run_idx,
                "run_total": len(TRAINING_RUN_SEEDS),
                "scenario_idx": scenario_idx,
                "scenario_total": len(MANUSCRIPT_SCENARIOS),
                "overall_idx": (
                    (run_idx - 1) * len(MANUSCRIPT_SCENARIOS)
                    + scenario_idx
                ),
                "overall_total": (
                    len(TRAINING_RUN_SEEDS)
                    * len(MANUSCRIPT_SCENARIOS)
                ),
                "scenario": scenario,
            }

            train_monitor = monitor_for_view(view)

            if scenario in central_pair:
                paired_fed = central_pair[scenario]
                exact_steps = int(
                    sum(
                        sum(m.values())
                        for m in fed_step_maps[paired_fed]
                    )
                )
                result = centralized_train(
                    build_model_fn,
                    initial_weights,
                    selected,
                    view["client_arrays"],
                    view["X_test"],
                    view["y_test"],
                    3,
                    BATCH_SIZE,
                    exact_steps,
                    view["class_weights"],
                    seed=seed + 10000,
                )
                central_fed_runtime_rows.append({
                    "run": run_idx,
                    "centralized": scenario,
                    "federated_pair": paired_fed,
                    "centralized_optimizer_steps": exact_steps,
                    "planned_federated_optimizer_steps": exact_steps,
                    "input_dim": input_dim,
                    "preprocessor_sha256": view[
                        "preprocessor_sha256"
                    ],
                    "initial_weights_sha256": initial_hash,
                })

            elif scenario == "Power-of-Choice":
                result = federated_train_power_of_choice(
                    build_model_fn=build_model_fn,
                    initial_weights=initial_weights,
                    candidate_clients=all_clients,
                    select_k=len(tadp_clients),
                    target_steps_per_round=tadp_target_per_round,
                    client_arrays=view["client_arrays"],
                    X_test=view["X_test"],
                    y_test=view["y_test"],
                    n_classes=3,
                    batch_size=BATCH_SIZE,
                    local_epochs=LOCAL_EPOCHS,
                    class_weights=view["class_weights"],
                    run_seed=seed,
                    train_monitor=train_monitor,
                    progress_context=progress_context,
                )
                selected = sorted(set(
                    cid
                    for rr in result["round_audit"]
                    for cid in str(
                        rr["selected_ids"]
                    ).split(";")
                    if cid
                ))

            else:
                selected_per_round = [
                    list(selected) for _ in range(NUM_ROUNDS_FL)
                ]
                exact_maps = fed_step_maps[scenario]
                result = federated_train(
                    build_model_fn,
                    initial_weights,
                    selected_per_round,
                    view["client_arrays"],
                    view["X_test"],
                    view["y_test"],
                    3,
                    BATCH_SIZE,
                    LOCAL_EPOCHS,
                    view["class_weights"],
                    run_seed=seed,
                    exact_step_maps=exact_maps,
                    equal_weight=False,
                    fedprox_mu=(
                        FEDPROX_MU if scenario == "FedProx" else 0.0
                    ),
                    train_monitor=train_monitor,
                    progress_context=progress_context,
                )

            row = result_row(
                run_idx,
                seed,
                scenario,
                result,
                initial_hash,
            )
            row["selected_client_ids"] = ";".join(selected)
            row["selected_client_count"] = (
                len(tadp_clients)
                if scenario == "Power-of-Choice"
                else len(selected)
            )
            row["preprocessing_fit_client_ids"] = ";".join(
                view["participating_clients"]
            )
            row["preprocessing_fit_client_count"] = int(
                view["participating_client_count"]
            )
            row["preprocessor_sha256"] = view[
                "preprocessor_sha256"
            ]
            row["input_dim"] = input_dim
            row["initialization_seed"] = seed
            row["fl_rounds"] = (
                0 if scenario in central_pair else NUM_ROUNDS_FL
            )
            row["hps_review_midpoint_0_5"] = float(
                REVIEW_MIDPOINT
            )
            row["option_A_post_selection_preprocessing"] = True
            row["frozen_evidence_assignment_seed"] = (
                FROZEN_EVIDENCE_ASSIGNMENT_SEED
            )
            row["experiment_version"] = EXPERIMENT_VERSION
            row["checkpoint_schema_version"] = int(
                CHECKPOINT_SCHEMA_VERSION
            )

            upsert_csv(
                row,
                PERF_CHECKPOINT,
                ["run", "scenario"],
            )
            write_scenario_checkpoint(
                EXPERIMENT_ROOT,
                run_idx,
                scenario,
                result,
            )
            mark_checkpoint_complete(
                CHECKPOINT_STATE,
                key,
                extra={
                    "experiment_version": EXPERIMENT_VERSION,
                    "checkpoint_schema_version": int(
                        CHECKPOINT_SCHEMA_VERSION
                    ),
                    "run": run_idx,
                    "scenario": scenario,
                },
            )
            completed.add(key)

            selection_rows.append({
                "run": run_idx,
                "scenario": scenario,
                "selected_clients": (
                    "DYNAMIC_PER_ROUND"
                    if scenario == "Power-of-Choice"
                    else ";".join(selected)
                ),
                "selected_count": (
                    len(tadp_clients)
                    if scenario == "Power-of-Choice"
                    else len(selected)
                ),
                "preprocessing_fit_clients": ";".join(
                    view["participating_clients"]
                ),
                "preprocessing_fit_client_count": int(
                    view["participating_client_count"]
                ),
                "preprocessor_sha256": view[
                    "preprocessor_sha256"
                ],
            })

            print(
                f"FINAL TEST | {scenario} | "
                f"Acc={row['accuracy']:.6f} | "
                f"F1={row['f1_macro']:.6f} | "
                f"AUC={row['roc_auc_ovr_macro']:.6f} | "
                f"input_dim={input_dim}"
            )

            if "model" in result:
                del result["model"]
            tf.keras.backend.clear_session()
            gc.collect()

    # ------------------------------------------------------------------
    # FINAL FAIL-CLOSED AUDITS
    # ------------------------------------------------------------------
    perf = pd.read_csv(PERF_CHECKPOINT)
    expected_rows = (
        len(TRAINING_RUN_SEEDS) * len(MANUSCRIPT_SCENARIOS)
    )
    if len(
        perf[["run", "scenario"]].drop_duplicates()
    ) != expected_rows:
        raise RuntimeError(
            f"Incomplete run: expected {expected_rows} results."
        )

    # Centralized/federated pairs must use identical post-selection transform,
    # input dimension, initialization, and optimizer-step budget.
    pair_audit_rows = []
    for run_idx in sorted(perf["run"].unique()):
        for central_name, fed_name in {
            "Naïve Centralized": "Vanilla FedAvg",
            "TADP Centralized": "TADP Federated",
        }.items():
            c = perf[
                (perf["run"].eq(run_idx))
                & perf["scenario"].eq(central_name)
            ].iloc[0]
            f = perf[
                (perf["run"].eq(run_idx))
                & perf["scenario"].eq(fed_name)
            ].iloc[0]
            pair_audit_rows.append({
                "run": int(run_idx),
                "centralized": central_name,
                "federated": fed_name,
                "same_input_dim": (
                    int(c["input_dim"]) == int(f["input_dim"])
                ),
                "same_preprocessor": (
                    str(c["preprocessor_sha256"])
                    == str(f["preprocessor_sha256"])
                ),
                "same_initial_weights": (
                    str(c["initial_weights_sha256"])
                    == str(f["initial_weights_sha256"])
                ),
                "exact_step_parity": (
                    int(c["optimizer_steps"])
                    == int(f["optimizer_steps"])
                ),
            })
    pair_audit = pd.DataFrame(pair_audit_rows)
    pair_audit.to_csv(
        EXPERIMENT_ROOT
        / "centralized_federated_runtime_parity_audit.csv",
        index=False,
    )
    if not pair_audit[
        [
            "same_input_dim",
            "same_preprocessor",
            "same_initial_weights",
            "exact_step_parity",
        ]
    ].all().all():
        raise RuntimeError(
            "Centralized/federated runtime parity audit failed."
        )

    # TADP vs matched selection controls: same K and total steps.
    # Each selection policy legitimately fits preprocessing from its own selected
    # cohort under Option A. Exact W0 equality is required only when input
    # dimensions are equal; otherwise same initialization seed is recorded.
    control_rows = []
    for run_idx in sorted(perf["run"].unique()):
        tadp_ref = perf[
            (perf["run"].eq(run_idx))
            & perf["scenario"].eq("TADP Federated")
        ].iloc[0]
        for baseline in [
            "Random-K",
            "DQ-only Federated",
            "Power-of-Choice",
        ]:
            b = perf[
                (perf["run"].eq(run_idx))
                & perf["scenario"].eq(baseline)
            ].iloc[0]
            same_dim = (
                int(tadp_ref["input_dim"]) == int(b["input_dim"])
            )
            same_w0_if_comparable = (
                str(tadp_ref["initial_weights_sha256"])
                == str(b["initial_weights_sha256"])
                if same_dim
                else np.nan
            )
            control_rows.append({
                "run": int(run_idx),
                "baseline": baseline,
                "tadp_k": int(tadp_ref["selected_client_count"]),
                "baseline_k": int(b["selected_client_count"]),
                "same_k": (
                    int(tadp_ref["selected_client_count"])
                    == int(b["selected_client_count"])
                ),
                "tadp_optimizer_steps": int(tadp_ref["optimizer_steps"]),
                "baseline_optimizer_steps": int(b["optimizer_steps"]),
                "exact_step_parity": (
                    int(tadp_ref["optimizer_steps"])
                    == int(b["optimizer_steps"])
                ),
                "same_initialization_seed": (
                    int(tadp_ref["initialization_seed"])
                    == int(b["initialization_seed"])
                ),
                "same_input_dim": bool(same_dim),
                "same_initial_weights_if_same_input_dim": (
                    same_w0_if_comparable
                ),
                "tadp_preprocessor_sha256": str(
                    tadp_ref["preprocessor_sha256"]
                ),
                "baseline_preprocessor_sha256": str(
                    b["preprocessor_sha256"]
                ),
                "option_A_note": (
                    "each selection policy fits scaler/vocabulary/class "
                    "weights only from its own participating/candidate cohort"
                ),
            })
    control_audit = pd.DataFrame(control_rows)
    control_audit.to_csv(
        EXPERIMENT_ROOT
        / "tadp_controlled_selection_budget_audit.csv",
        index=False,
    )
    if not control_audit[
        ["same_k", "exact_step_parity", "same_initialization_seed"]
    ].all().all():
        raise RuntimeError(
            "Matched selection-control budget audit failed."
        )

    # Verify the TADP preprocessing cohort remained frozen across all runs.
    tadp_perf = perf[
        perf["scenario"].isin(
            ["TADP Centralized", "TADP Federated"]
        )
    ].copy()
    expected_tadp_fit = ";".join(
        scenario_views["TADP Federated"][
            "participating_clients"
        ]
    )
    if not tadp_perf[
        "preprocessing_fit_client_ids"
    ].astype(str).eq(expected_tadp_fit).all():
        raise RuntimeError(
            "TADP preprocessing cohort changed across runs."
        )

    pd.DataFrame(selection_rows).drop_duplicates().to_csv(
        EXPERIMENT_ROOT / "selection_audit.csv",
        index=False,
    )
    pd.DataFrame(central_fed_runtime_rows).drop_duplicates().to_csv(
        EXPERIMENT_ROOT / "centralized_federated_step_parity.csv",
        index=False,
    )

    # ------------------------------------------------------------------
    # PUBLICATION TABLES
    # ------------------------------------------------------------------
    perf = perf.sort_values(
        ["run", "scenario"]
    ).reset_index(drop=True)
    perf.to_csv(
        EXPERIMENT_ROOT / "performance_metrics.csv",
        index=False,
    )
    summary = summarize_runs_with_ci(perf)
    summary.to_csv(
        EXPERIMENT_ROOT / "performance_summary_mean_sd_ci95.csv",
        index=False,
    )
    paired = paired_tadp_randomk_statistics(perf)
    paired.to_csv(
        EXPERIMENT_ROOT / "paired_tadp_randomk_statistics.csv",
        index=False,
    )

    resource_cols = [
        "run", "seed", "scenario",
        "runtime_s",
        "energy_wh_per_watt",
        "reference_avg_compute_power_w",
        "energy_wh", "energy_kwh",
        "reference_grid_co2_kg_per_kwh", "co2_kg",
        "communication_payload_bytes", "communication_mb", "communication_mib",
        "ram_peak_mb", "ram_delta_mb",
        "reference_electricity_usd_per_kwh",
        "reference_communication_usd_per_mb",
        "energy_cost_usd", "communication_cost_usd",
        "total_estimated_cost_usd",
        "optimizer_steps", "selected_client_count",
        "input_dim", "preprocessor_sha256",
    ]
    perf[
        [c for c in resource_cols if c in perf.columns]
    ].to_csv(
        EXPERIMENT_ROOT / "resource_metrics.csv",
        index=False,
    )

    policy = {
        "experiment_version": EXPERIMENT_VERSION,
        "experiment": "A1 main benchmark",
        "K": NUM_CLIENTS,
        "scenarios": MANUSCRIPT_SCENARIOS,
        "training_seeds": TRAINING_RUN_SEEDS,
        "number_of_runs": len(TRAINING_RUN_SEEDS),
        "fl_rounds": NUM_ROUNDS_FL,
        "local_epochs": LOCAL_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "fedprox_mu": FEDPROX_MU,
        "dirichlet_alpha": DIRICHLET_ALPHA,
        "total_factors": 28,
        "DQ_raw_train_factors": 8,
        "HPS_weights": WEIGHTS_PSCORE_DEFAULT,
        "evidence_validity_gate": {
            "required_factor_count": 28,
            "rule": "all_scores_present_finite_and_strictly_gt_0",
            "factor_specific_minima_used": False,
        },
        "dimension_min_floor": DIMENSION_MIN_FLOOR,
        "HPS_reject_threshold": GOOD_CUT,
        "HPS_direct_accept_threshold": HIGH_CUT,
        "critical_review_dimensions": list(
            CRITICAL_REVIEW_DIMS
        ),
        "critical_review_dimension_names": {
            d: DIMENSION_NAMES[d]
            for d in CRITICAL_REVIEW_DIMS
        },
        "HPS_review_weights": CRITICAL_REVIEW_WEIGHTS,
        "HPS_review_definition": (
            "original HPS weights renormalized over dim1,dim2,dim5,dim6"
        ),
        "HPS_review_midpoint": REVIEW_MIDPOINT,
        "HPS_review_accept_rule": (
            "greater than or equal to midpoint; equality accepts"
        ),
        "documentation_and_timeliness_in_main_HPS": True,
        "documentation_and_timeliness_in_HPS_review": False,
        "option_A_pipeline": (
            "raw TRAIN shards -> DQ/governance -> Accept/Reject -> "
            "fit scaler/vocabulary/class weights from participating TRAIN "
            "cohort only -> train -> held-out TEST evaluation"
        ),
        "TADP_rejected_clients_in_TADP_preprocessing": False,
        "scenario_specific_post_selection_preprocessing": True,
        "Power_of_Choice_preprocessing_universe": (
            "all clients because all are candidates for dynamic round selection"
        ),
        "test_semantics": (
            "held-out final evaluation only; never used for governance, "
            "selection, preprocessing, class weights, or training"
        ),
        "operational_accounting": {
            "runtime": "measured with time.perf_counter",
            "communication": (
                "application-level payload bytes derived from actual model "
                "weight arrays / model-ready centralized arrays; no 12% "
                "protocol-overhead multiplier; not packet/network capture"
            ),
            "reference_avg_compute_power_w": REFERENCE_AVG_COMPUTE_POWER_W,
            "energy_formula": "E_Wh = P_ref_W * runtime_s / 3600",
            "energy_is_estimate_not_measurement": True,
            "reference_grid_co2_kg_per_kwh": REFERENCE_GRID_CO2_KG_PER_KWH,
            "co2_formula": "CO2_kg = E_kWh * grid_intensity_kg_per_kWh",
            "reference_electricity_usd_per_kwh": REFERENCE_ELECTRICITY_USD_PER_KWH,
            "reference_communication_usd_per_mb": REFERENCE_COMMUNICATION_USD_PER_MB,
            "costs_are_reference_scenario_assumptions": True,
            "ledger_size": "measured final UTF-8 CSV bytes on disk",
            "ledger_write_time": (
                "application-level serialization + flush + fsync request; "
                "not raw physical-device latency"
            ),
        },
    }
    atomic_write_json(
        policy,
        EXPERIMENT_ROOT / "policy_and_experiment_design.json",
    )

    print_banner("FINAL PERFORMANCE SUMMARY")
    cols = [
        "scenario", "n_runs", "accuracy_mean", "accuracy_sd",
        "f1_macro_mean", "f1_macro_sd",
        "roc_auc_ovr_macro_mean", "roc_auc_ovr_macro_sd",
        "runtime_s_mean", "communication_mb_mean",
        "ram_peak_mb_mean",
    ]
    print(
        summary[
            [c for c in cols if c in summary.columns]
        ].to_string(index=False)
    )

    print_banner("FINAL FAIL-CLOSED AUDITS")
    print("Global TRAIN/TEST leakage audit: PASS")
    print("28-factor evidence-validity policy audit: PASS")
    print("Dimension-floor policy audit: PASS")
    print("4-dimension HPS_review policy audit: PASS")
    print("Option A governance-before-preprocessing audit: PASS")
    print("TADP rejected clients excluded from TADP preprocessing: PASS")
    print("Centralized/federated pair preprocessing + W0 + step parity: PASS")
    print("TADP/Random-K matched K, rounds, and exact realized step budget: PASS")
    print("DQ-only matched TADP K, rounds, and exact realized step budget using DQ-only ranking: PASS")
    print("Power-of-Choice matched K, rounds, and exact realized step budget: PASS")
    print(
        f"Completed {len(MANUSCRIPT_SCENARIOS)} headline scenarios x "
        f"{len(TRAINING_RUN_SEEDS)} seeds."
    )

    return EXPERIMENT_ROOT


if __name__ == "__main__":


    finished_root=main()
    package_and_download_results(finished_root,EXPERIMENT_VERSION)


✅ Reusing dataset already cached by the separate upload cell: /content/data_cache/diabetes_130US.csv
Google Drive checkpoint mount unavailable: Error: credential propagation was unsuccessful
Local checkpoint root: /content/TADP_EXPERIMENT_A1_TADP-A1-FINAL-28F-DIMFLOOR-4D-HPSREVIEW-OPTIONA-STEPPARITY-K10-5SEED
✅ Reusing existing Diabetes dataset: /content/data_cache/diabetes_130US.csv

TADP-A1-FINAL-28F-DIMFLOOR-4D-HPSREVIEW-OPTIONA-STEPPARITY-K10-5SEED
Training seeds: [42, 142, 242, 342, 442]
K=10 raw TRAIN clients | FL rounds=4
FINAL TADP policy: 28-factor validity gate -> 6-dimension floor -> 6-dimension HPS -> 4-critical-dimension HPS_review for borderline cases.
Evidence validity: all 28 scores must be present, finite, and > 0.0.
Dimension floor: every dimension >= 2.5/5.
Main HPS: <3.0=Reject; [3.0,3.5)=Review; >=3.5=Direct Accept.
Review dimensions: dim1:Source Reliability, dim2:Data Quality and Health, dim5:Regulatory and Compliance Alignment, dim6:Context and Usage Constraints


INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmp0hjb002n' for ephemeral docs site


Calculating Metrics:   0%|          | 0/223 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/223 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/223 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/223 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/223 [00:00<?, ?it/s]


DOMAIN ADMISSION POLICY — 28-FACTOR VALIDITY + HPS + CRITICAL-DIMENSION REVIEW
Evidence-validity gate: all 28 factor scores must be present, finite, and strictly >0.
Dimension floor: every averaged dimension must be >= 2.5/5.
Main HPS: <3.0=Reject; [3.0,3.5)=Automated Review; >=3.5=Direct Accept.
Automated Review critical dimensions: dim1=Source Reliability, dim2=Data Quality and Health, dim5=Regulatory and Compliance Alignment, dim6=Context and Usage Constraints
Renormalized critical weights: {"dim1": 0.3125, "dim2": 0.18749999999999997, "dim5": 0.37499999999999994, "dim6": 0.125}
HPS_review >= 3.25/5 -> Accept after Review; otherwise Reject.
Human reviewer role: verify documentary evidence only; the server performs scoring and admission automatically.

RAW TRAIN-SHARD DATA-QUALITY FACTORS — 8 FACTORS
client  completeness  duplication_rate  value_validity_error_rate  type_consistency  label_integrity  feature_distribution_consistency  feature_category_coverage  structural_constraint_

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>